# **Step 1 — Food Loss and Waste Data Exploration & Validation**

**Purpose:** profile the FAO Food Loss and Waste dataset, identify data-quality risks, validate geographic and commodity codes against authoritative references, and produce reviewable reports without silently changing the source data.

## ***Import libraries***

In [165]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt



## ***Read the dataset***

In [166]:
# Locate the source CSV without hard-coding one working directory.
# You can override this with an environment variable named FOOD_LOSS_DATA_PATH.
import os
from pathlib import Path
import pandas as pd
from IPython.display import display

DATA_FILENAME = "World-food-waste.csv"
configured_path = os.getenv("FOOD_LOSS_DATA_PATH")

candidate_paths = []
if configured_path:
    candidate_paths.append(Path(configured_path).expanduser())

cwd = Path.cwd()
candidate_paths.extend([
    cwd / DATA_FILENAME,
    cwd / "data-set" / DATA_FILENAME,
    cwd.parent / "data-set" / DATA_FILENAME,
    cwd / ".." / "data-set" / DATA_FILENAME,
    cwd / "data" / DATA_FILENAME,
    cwd.parent / "data" / DATA_FILENAME,
])

DATA_PATH = next((candidate.resolve() for candidate in candidate_paths
                  if candidate.exists() and candidate.is_file()), None)

if DATA_PATH is None:
    checked = "\n".join(f"- {candidate}" for candidate in candidate_paths)
    raise FileNotFoundError(
        f"Could not locate {DATA_FILENAME}. Set FOOD_LOSS_DATA_PATH or place "
        f"the CSV in one of the expected locations:\n{checked}"
    )

world_food_loss = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded: {DATA_PATH}")
print(f"Shape: {world_food_loss.shape}")
display(world_food_loss.head(3))

Loaded: D:\My_projects\data-engineer\waste-food-project\data-set\World-food-waste.csv
Shape: (31465, 18)


,m49_code,country,region,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,food_supply_stage,treatment,cause_of_loss,sample_size,method_data_collection,reference,url,notes
0,1,World,NaN,0111,Wheat,2022,1.0,1%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is wheat flour
1,1,World,NaN,0112,Maize (corn),2022,0.5,0.5%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is maize flour.
2,1,World,NaN,0112,Maize (corn),2022,2.0,2%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is Maize (corn) grits.


## **Understand Columns**

![Food Loss Image](../images/columns.jpg)

## **Expected Problems**

### Validation principles
- **Profile first:** inspect data types, missingness, uniqueness, duplicates, and distributions.
- **Validate against references:** distinguish invalid codes from valid codes paired with the wrong label.
- **Flag, do not silently repair:** normalization is for comparison; original values remain unchanged.
- **Keep provenance:** record the reference source, retrieval date, rule, and review outcome.
- **Treat anomalies as review candidates:** outliers, rare values, and missing references are not automatically errors.

![Food Loss Image](../images/expected-problems.jpg)

## ***Data set overview***


In [167]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print("Shape:", world_food_loss.shape)
print("Number of rows:", world_food_loss.shape[0])
print("Number of columns:", world_food_loss.shape[1])
print("Memory usage (MB):", round(world_food_loss.memory_usage(deep=True).sum() / (1024 ** 2), 2))

display(world_food_loss.sample(min(5, len(world_food_loss)), random_state=42))

DATASET OVERVIEW
Shape: (31465, 18)
Number of rows: 31465
Number of columns: 18
Memory usage (MB): 25.66


,m49_code,country,region,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,food_supply_stage,treatment,cause_of_loss,sample_size,method_data_collection,reference,url,notes
27126,826,United Kingdom of Great Britain and Northern I...,NaN,01312,Bananas,2014,1.50,0–3%,NaN,Packaging,Packing,NaN,NaN,NaN,Survey,"Carlos Mena, Leon A. Terry, Adrian Williams; L...",NaN,NaN
17402,51,Armenia,NaN,01199.90,Other cereals n.e.,2012,4.17,4.17%,600,NaN,Whole supply chain,NaN,NaN,NaN,FAO's annual Agriculture Production Questionna...,FAO Sources,NaN,NaN
2320,180,Democratic Republic of the Congo,NaN,0118,Millet,2017,2.50,2.5,NaN,Winnowing,Farm,NaN,NaN,NaN,Modelled Estimates,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...,NaN
25378,768,Togo,NaN,0114,Sorghum,2020,3.60,3.6,NaN,"Shelling, Threshing",NaN,NaN,NaN,NaN,Modelled Estimates,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...,NaN
10883,384,Côte d'Ivoire,NaN,0112,Maize (corn),2021,4.00,4,NaN,Drying,Farm,NaN,NaN,NaN,Modelled Estimates,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...,NaN


## ***Analysis Columns***

In [168]:
print("COLUMN NAMES")
print(world_food_loss.columns.tolist())

print("\nDATA TYPES")
print(world_food_loss.dtypes)

print("\nDATASET INFO")
world_food_loss.info()

print("\nNUMERIC SUMMARY")
display(world_food_loss.describe().T)

print("\nALL COLUMNS SUMMARY")
display(world_food_loss.describe(include="all").T)

COLUMN NAMES
['m49_code', 'country', 'region', 'cpc_code', 'commodity', 'year', 'loss_percentage', 'loss_percentage_original', 'loss_quantity', 'activity', 'food_supply_stage', 'treatment', 'cause_of_loss', 'sample_size', 'method_data_collection', 'reference', 'url', 'notes']

DATA TYPES
m49_code                      int64
country                         str
region                          str
cpc_code                        str
commodity                       str
year                          int64
loss_percentage             float64
loss_percentage_original        str
loss_quantity                   str
activity                        str
food_supply_stage               str
treatment                       str
cause_of_loss                   str
sample_size                     str
method_data_collection          str
reference                       str
url                             str
notes                           str
dtype: object

DATASET INFO
<class 'pandas.DataFrame'>
RangeInd

,count,mean,std,min,25%,50%,75%,max
m49_code,31465.0,471.301637,260.948893,1.0,233.00000,454.00,710.00,894.0
year,31465.0,2010.610774,6.220568,2000.0,2005.00000,2011.00,2016.00,2026.0
loss_percentage,31465.0,4.683186,6.359491,0.0,1.49833,2.65,4.55,65.0



ALL COLUMNS SUMMARY


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
m49_code,31465.0,NaN,NaN,NaN,471.301637,260.948893,1.0,233.0,454.0,710.0,894.0
country,31465,159,United States of America,1716,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,2036,378,Western Cape Province,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cpc_code,31465,196,0112,6090,NaN,NaN,NaN,NaN,NaN,NaN,NaN
commodity,31465,213,Maize (corn),6090,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,31465.0,NaN,NaN,NaN,2010.610774,6.220568,2000.0,2005.0,2011.0,2016.0,2026.0
loss_percentage,31465.0,NaN,NaN,NaN,4.683186,6.359491,0.0,1.49833,2.65,4.55,65.0
loss_percentage_original,31465,5995,2.5,2309,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loss_quantity,5050,3429,1000,74,NaN,NaN,NaN,NaN,NaN,NaN,NaN
activity,24095,180,Storage,6072,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## ***Analysis Missing values in Data-set***

In [169]:
missing_report = pd.DataFrame({
    "missing_count": world_food_loss.isna().sum(),
    "missing_percentage": world_food_loss.isna().mean().mul(100),
    "non_missing_count": world_food_loss.notna().sum(),
    "unique_non_null": world_food_loss.nunique(dropna=True)
})

missing_report = missing_report.sort_values(
    by="missing_percentage",
    ascending=False
)

display(missing_report)

,missing_count,missing_percentage,non_missing_count,unique_non_null
cause_of_loss,30367,96.510408,1098,406
notes,29931,95.124742,1534,586
treatment,29678,94.320674,1787,764
sample_size,29603,94.082314,1862,266
region,29429,93.529318,2036,378
loss_quantity,26415,83.950421,5050,3429
url,9342,29.690132,22123,948
activity,7370,23.422851,24095,180
food_supply_stage,3391,10.777054,28074,20
reference,73,0.232004,31392,328


### ***Show "NA", "N/A", "NULL", "NONE", "UNKNOWN", "MISSING", "-" in string data-types***

In [170]:

missing_tokens = {
    "NA", "N/A", "NULL", "NONE",
    "UNKNOWN", "MISSING", "-", "NAN"
}

text_df = world_food_loss.select_dtypes(
    include=["object", "string"]
)

results = []

for col in text_df.columns:
    values = text_df[col].astype("string").str.strip()

    empty_count = values.eq("").sum()

    token_count = values.str.upper().isin(
        missing_tokens
    ).sum()

    actual_missing_count = values.isna().sum()

    results.append({
        "column": col,
        "empty_string_count": int(empty_count),
        "missing_token_count": int(token_count),
        "actual_missing_count": int(actual_missing_count)
    })

missing_report = (
    pd.DataFrame(results)
    .set_index("column")
    .sort_values(
        "actual_missing_count",
        ascending=False
    )
)

display(missing_report)

,empty_string_count,missing_token_count,actual_missing_count
column,,,
cause_of_loss,0,0,30367
notes,0,0,29931
treatment,0,0,29678
sample_size,0,6,29603
region,0,0,29429
loss_quantity,0,0,26415
url,0,0,9342
activity,0,0,7370
food_supply_stage,0,0,3391


## ***Show duplicated and unique data***

In [171]:
unique_report = pd.DataFrame({
    "unique_count": world_food_loss.nunique(dropna=True),
    "unique_with_nan": world_food_loss.nunique(dropna=False),
    "dtype": world_food_loss.dtypes.astype(str),
    "is_constant": world_food_loss.nunique(dropna=True).le(1)
})

display(unique_report.sort_values("unique_count"))

,unique_count,unique_with_nan,dtype,is_constant
method_data_collection,11,12,str,False
food_supply_stage,20,21,str,False
year,27,27,int64,False
m49_code,157,157,int64,False
country,159,159,str,False
activity,180,181,str,False
cpc_code,196,196,str,False
commodity,213,213,str,False
sample_size,266,267,str,False
reference,328,329,str,False


In [172]:
# Select categorical and text columns
categorical_cols = world_food_loss.select_dtypes(
    include=["object", "string", "category"]
).columns

# Store column profiling results
column_profile = []

for col in categorical_cols:
    column_profile.append({
        "Column Name": col,
        "Data Type": str(world_food_loss[col].dtype),
        "Unique Values": world_food_loss[col].nunique(dropna=True),
        "Most Frequent Value": (
            world_food_loss[col].mode(dropna=True).iloc[0]
            if not world_food_loss[col].mode(dropna=True).empty
            else None
        ),
        "Most Frequent Count": (
            world_food_loss[col].value_counts(dropna=True).iloc[0]
            if not world_food_loss[col].value_counts(dropna=True).empty
            else 0
        )
    })

# Display the summary table
profile_df = pd.DataFrame(column_profile)

display(
    profile_df.style
    .set_caption("Categorical Columns Profile")
    .format(precision=2)
    .background_gradient(subset=["Unique Values"], cmap="Blues")
    .set_properties(**{
        "text-align": "left",
        "padding": "8px"
    })
)


,Column Name,Data Type,Unique Values,Most Frequent Value,Most Frequent Count
0,country,str,159,United States of America,1716
1,region,str,378,Western Cape Province,59
2,cpc_code,str,196,0112,6090
3,commodity,str,213,Maize (corn),6090
4,loss_percentage_original,str,5995,2.5,2309
5,loss_quantity,str,3429,1000,74
6,activity,str,180,Storage,6072
7,food_supply_stage,str,20,Farm,10103
8,treatment,str,764,Aman rice,57
9,cause_of_loss,str,406,Physical Losses,87


In [173]:
print("Fully duplicated rows:", world_food_loss.duplicated().sum())

print(
    "Duplicated rows including all copies:",
    world_food_loss.duplicated(keep=False).sum()
)

duplicates = world_food_loss[
    world_food_loss.duplicated(keep=False)
].sort_values(by=world_food_loss.columns.tolist())

display(duplicates.head(20))

Fully duplicated rows: 180
Duplicated rows including all copies: 339


,m49_code,country,region,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,food_supply_stage,treatment,cause_of_loss,sample_size,method_data_collection,reference,url,notes
16249,50,Bangladesh,NaN,0113,Rice,2010,1.30,1.3,NaN,NaN,Processing,NaN,NaN,NaN,Modelled Estimates,Mst. Esmat Ara Begum; Mohammad Ismail Hossain;...,http://www.setscholars.org/index.php/ijarbae,NaN
16252,50,Bangladesh,NaN,0113,Rice,2010,1.30,1.3,NaN,NaN,Processing,NaN,NaN,NaN,Modelled Estimates,Mst. Esmat Ara Begum; Mohammad Ismail Hossain;...,http://www.setscholars.org/index.php/ijarbae,NaN
16123,50,Bangladesh,NaN,01312,Bananas,2010,3.20,3.2,NaN,NaN,Retail,NaN,NaN,100,Survey,Md Kamrul Hassan; Bishan Lal Das Chowdhury; Na...,http://www.measurepostharvestlosses.com/sites/...,NaN
16125,50,Bangladesh,NaN,01312,Bananas,2010,3.20,3.2,NaN,NaN,Retail,NaN,NaN,100,Survey,Md Kamrul Hassan; Bishan Lal Das Chowdhury; Na...,http://www.measurepostharvestlosses.com/sites/...,NaN
16121,50,Bangladesh,NaN,01312,Bananas,2010,5.10,5.1,NaN,NaN,Trader,NaN,NaN,100,Survey,Md Kamrul Hassan; Bishan Lal Das Chowdhury; Na...,http://www.measurepostharvestlosses.com/sites/...,NaN
16122,50,Bangladesh,NaN,01312,Bananas,2010,5.10,5.1,NaN,NaN,Trader,NaN,NaN,100,Survey,Md Kamrul Hassan; Bishan Lal Das Chowdhury; Na...,http://www.measurepostharvestlosses.com/sites/...,NaN
16120,50,Bangladesh,NaN,01312,Bananas,2010,7.70,7.7,NaN,NaN,Farm,NaN,NaN,100,Survey,Md Kamrul Hassan; Bishan Lal Das Chowdhury; Na...,http://www.measurepostharvestlosses.com/sites/...,NaN
16126,50,Bangladesh,NaN,01312,Bananas,2010,7.70,7.7,NaN,NaN,Farm,NaN,NaN,100,Survey,Md Kamrul Hassan; Bishan Lal Das Chowdhury; Na...,http://www.measurepostharvestlosses.com/sites/...,NaN
16124,50,Bangladesh,NaN,01312,Bananas,2010,8.60,8.6,NaN,NaN,Wholesale,NaN,NaN,100,Survey,Md Kamrul Hassan; Bishan Lal Das Chowdhury; Na...,http://www.measurepostharvestlosses.com/sites/...,NaN
16130,50,Bangladesh,NaN,01312,Bananas,2010,8.60,8.6,NaN,NaN,Wholesale,NaN,NaN,100,Survey,Md Kamrul Hassan; Bishan Lal Das Chowdhury; Na...,http://www.measurepostharvestlosses.com/sites/...,NaN


## ***Describe numeric data***

In [174]:
numeric_cols = world_food_loss.select_dtypes(
    include=np.number
).columns

numeric_report = pd.DataFrame(index=numeric_cols)

numeric_report["missing"] = world_food_loss[numeric_cols].isna().sum()
numeric_report["unique"] = world_food_loss[numeric_cols].nunique()
numeric_report["min"] = world_food_loss[numeric_cols].min()
numeric_report["max"] = world_food_loss[numeric_cols].max()
numeric_report["mean"] = world_food_loss[numeric_cols].mean()
numeric_report["median"] = world_food_loss[numeric_cols].median()
numeric_report["std"] = world_food_loss[numeric_cols].std()
numeric_report["negative_count"] = world_food_loss[numeric_cols].lt(0).sum()

display(numeric_report)

,missing,unique,min,max,mean,median,std,negative_count
m49_code,0,157,1.0,894.0,471.301637,454.00,260.948893,0
year,0,27,2000.0,2026.0,2010.610774,2011.00,6.220568,0
loss_percentage,0,4629,0.0,65.0,4.683186,2.65,6.359491,0


## ***Show ouliers***

In [175]:
outlier_rows = []

for col in numeric_cols:
    series = world_food_loss[col].dropna()

    if series.empty:
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (world_food_loss[col] < lower_bound) | (
        world_food_loss[col] > upper_bound
    )

    outlier_rows.append({
        "column": col,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": int(mask.sum()),
        "outlier_percentage": round(mask.mean() * 100, 2)
    })

outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

,column,lower_bound,upper_bound,outlier_count,outlier_percentage
0,m49_code,-482.500000,1425.500000,0,0.0
1,year,1988.500000,2032.500000,0,0.0
2,loss_percentage,-3.079175,9.127505,3555,11.3


## ***Show final columns***

In [176]:

columns_df = pd.DataFrame({
    "Column Number": range(1, len(world_food_loss.columns) + 1),
    "Column Name": world_food_loss.columns.tolist(),
    "Data Type": world_food_loss.dtypes.astype(str).values
})

display(columns_df)


,Column Number,Column Name,Data Type
0,1,m49_code,int64
1,2,country,str
2,3,region,str
3,4,cpc_code,str
4,5,commodity,str
5,6,year,int64
6,7,loss_percentage,float64
7,8,loss_percentage_original,str
8,9,loss_quantity,str
9,10,activity,str


## ***Check on columns***

### **Check on m49_code**

In [177]:
# Country-to-M49 code inventory; create the output folder first.
from pathlib import Path

m49_report_dir = Path("reports") / "m49"
m49_report_dir.mkdir(parents=True, exist_ok=True)

res = world_food_loss.groupby("country", dropna=False)["m49_code"].unique()
res.to_csv(m49_report_dir / "country_m49_codes.csv")

print(f"Saved: {(m49_report_dir / 'country_m49_codes.csv').resolve()}")

Saved: D:\My_projects\data-engineer\waste-food-project\notebooks\reports\m49\country_m49_codes.csv


#### **Repeated codes in diff countries**

In [178]:
code_names = (
    world_food_loss[['m49_code', 'country']]
    .dropna()
    .drop_duplicates()
)

repeated_codes = code_names[
    code_names.duplicated('m49_code', keep=False)
].sort_values('m49_code')

display(repeated_codes)

,m49_code,country
2215,158,"Taiwan, Province of China"
2217,158,"China,Taiwan"
27050,826,United Kingdom of Great Britain and Northern I...
27086,826,United Kingdom


#### **Missing values , datatype and unique values**

In [179]:
print("Missing values:", world_food_loss['m49_code'].isna().sum())
print("Data type:", world_food_loss['m49_code'].dtype)
print("Unique codes:", world_food_loss['m49_code'].nunique())

Missing values: 0
Data type: int64
Unique codes: 157


#### **Same country has many codes ?**

In [180]:
country_codes = (
    world_food_loss.groupby('country')['m49_code']
    .nunique(dropna=True)
)

multiple_codes = country_codes[country_codes > 1]

display(multiple_codes)

Series([], Name: m49_code, dtype: int64)

#### **Check invalid codes**

In [181]:
import requests
from io import StringIO

# 1) The three reference links
links = {
    "UN_M49_Overview":
        "https://unstats.un.org/unsd/methodology/m49/overview/",

    "UN_M49_Methodology":
        "https://unstats.un.org/unsd/methodology/m49/",

    "GitHub_ISO_Regional_Codes":
        "https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv",
}

def normalize_name(value):
    """Normalize names for comparison."""
    if pd.isna(value):
        return ""
    return " ".join(str(value).strip().casefold().split())

def normalize_code(value):
    """Normalize numeric codes to 3-digit strings."""
    if pd.isna(value):
        return ""

    value = str(value).strip()

    # Handle codes read as values such as 15.0
    if value.endswith(".0"):
        value = value[:-2]

    if value.isdigit():
        return value.zfill(3)

    return value


# 2) Read the official UN overview table
response = requests.get(links["UN_M49_Overview"], timeout=30)
response.raise_for_status()

un_tables = pd.read_html(StringIO(response.text))

un_reference = next(
    table for table in un_tables
    if "Country or Area" in table.columns
    and "M49 Code" in table.columns
)

# Country/area name -> country/area M49 code
country_reference = pd.DataFrame({
    "name": un_reference["Country or Area"],
    "code": un_reference["M49 Code"],
    "source": "UN M49 Overview",
    "record_type": "country_or_area",
})

# Geographic aggregate name -> its corresponding geographic code
region_columns = [
    ("Global Name", "Global Code", "global"),
    ("Region Name", "Region Code", "region"),
    ("Sub-region Name", "Sub-region Code", "sub_region"),
    (
        "Intermediate Region Name",
        "Intermediate Region Code",
        "intermediate_region",
    ),
]

region_frames = []

for name_col, code_col, record_type in region_columns:
    if name_col in un_reference.columns and code_col in un_reference.columns:
        region_frames.append(pd.DataFrame({
            "name": un_reference[name_col],
            "code": un_reference[code_col],
            "source": "UN M49 Overview",
            "record_type": record_type,
        }))

un_regions = pd.concat(region_frames, ignore_index=True)

# 3) Read the second UN page and the GitHub CSV
# Keep the raw downloaded data so we can inspect the actual columns.
methodology_response = requests.get(
    links["UN_M49_Methodology"], timeout=30
)
methodology_response.raise_for_status()

methodology_tables = pd.read_html(StringIO(methodology_response.text))

github_response = requests.get(
    links["GitHub_ISO_Regional_Codes"], timeout=30
)
github_response.raise_for_status()

github_reference = pd.read_csv(StringIO(github_response.text), dtype=str)

# GitHub uses ISO numeric country-code and separate regional-code columns.
github_frames = [
    pd.DataFrame({
        "name": github_reference["name"],
        "code": github_reference["country-code"],
        "source": "GitHub",
        "record_type": "country_or_area",
    }),
    pd.DataFrame({
        "name": github_reference["region"],
        "code": github_reference["region-code"],
        "source": "GitHub",
        "record_type": "region",
    }),
    pd.DataFrame({
        "name": github_reference["sub-region"],
        "code": github_reference["sub-region-code"],
        "source": "GitHub",
        "record_type": "sub_region",
    }),
    pd.DataFrame({
        "name": github_reference["intermediate-region"],
        "code": github_reference["intermediate-region-code"],
        "source": "GitHub",
        "record_type": "intermediate_region",
    }),
]

github_all = pd.concat(github_frames, ignore_index=True)

# 4) Combine reference pairs from all three links
# The methodology page is also searched for tables containing
# a country/area name and an M49 code.
methodology_frames = []

for table in methodology_tables:
    lower_columns = {
        str(col).strip().casefold(): col for col in table.columns
    }

    name_col = lower_columns.get("country or area")
    code_col = lower_columns.get("m49 code")

    if name_col is None:
        name_col = lower_columns.get("country or area")
    if code_col is None:
        code_col = lower_columns.get("m49 code")

    if name_col is not None and code_col is not None:
        methodology_frames.append(pd.DataFrame({
            "name": table[name_col],
            "code": table[code_col],
            "source": "UN M49 Methodology",
            "record_type": "country_or_area_or_region",
        }))

methodology_reference = (
    pd.concat(methodology_frames, ignore_index=True)
    if methodology_frames
    else pd.DataFrame(
        columns=["name", "code", "source", "record_type"]
    )
)

reference = pd.concat(
    [
        country_reference,
        un_regions,
        methodology_reference,
        github_all,
    ],
    ignore_index=True,
)

# 5) Normalize reference data
reference["name_key"] = reference["name"].map(normalize_name)
reference["code_key"] = reference["code"].map(normalize_code)

reference = reference[
    (reference["name_key"] != "")
    & (reference["code_key"] != "")
].drop_duplicates(
    subset=["name_key", "code_key", "source", "record_type"]
)

# 6) Normalize the unmatched records from your dataset
unmatched = world_food_loss[["country", "m49_code"]].drop_duplicates().copy()

unmatched["name_key"] = unmatched["country"].map(normalize_name)
unmatched["code_key"] = unmatched["m49_code"].map(normalize_code)

# 7) Check whether each exact name + code pair exists in each source
results = []

for _, row in unmatched.iterrows():
    matches = reference[
        (reference["name_key"] == row["name_key"])
        & (reference["code_key"] == row["code_key"])
    ]

    matched_sources = sorted(matches["source"].unique().tolist())

    results.append({
        "country": row["country"],
        "m49_code": row["code_key"],
        "found_in": ", ".join(matched_sources) if matched_sources else "",
        "match_count": len(matches),
        "status": (
            "FOUND"
            if not matches.empty
            else "NOT FOUND - REVIEW"
        ),
    })

validation = pd.DataFrame(results)

# Show only pairs that were not found in any reference
still_unmatched = validation[
    validation["status"] == "NOT FOUND - REVIEW"
]

print("All checked pairs:")
display(validation.sort_values(["status", "country"]))

print("\nStill unmatched after checking all three links:")
display(still_unmatched)

# Optional: inspect the exact reference records for a specific name
# Example:
# display(reference[reference["name_key"] == normalize_name("Taiwan, Province of China")])

# Optional: save the reports
validation.to_csv("../report/m49/m49_validation_all.csv", index=False)
still_unmatched.to_csv("../report/m49/m49_still_unmatched.csv", index=False)


All checked pairs:


,country,m49_code,found_in,match_count,status
67,Afghanistan,004,"GitHub, UN M49 Methodology, UN M49 Overview",3,FOUND
25,Africa,002,"GitHub, UN M49 Methodology, UN M49 Overview",3,FOUND
146,Albania,008,"GitHub, UN M49 Methodology, UN M49 Overview",3,FOUND
6,Algeria,012,"GitHub, UN M49 Methodology, UN M49 Overview",3,FOUND
35,Angola,024,"GitHub, UN M49 Methodology, UN M49 Overview",3,FOUND
...,...,...,...,...,...
98,Netherlands,528,,0,NOT FOUND - REVIEW
42,Palestine,275,,0,NOT FOUND - REVIEW
149,The former Yugoslav Republic of Macedonia,807,,0,NOT FOUND - REVIEW
145,Turkey,792,,0,NOT FOUND - REVIEW



Still unmatched after checking all three links:


,country,m49_code,found_in,match_count,status
19,"China,Taiwan",158,,0,NOT FOUND - REVIEW
42,Palestine,275,,0,NOT FOUND - REVIEW
98,Netherlands,528,,0,NOT FOUND - REVIEW
145,Turkey,792,,0,NOT FOUND - REVIEW
149,The former Yugoslav Republic of Macedonia,807,,0,NOT FOUND - REVIEW
152,United Kingdom,826,,0,NOT FOUND - REVIEW


In [182]:


# Known name variations and their current official UN M49 names.
name_mapping = {
    "Palestine": "State of Palestine",
    "Netherlands": "Netherlands (Kingdom of the)",
    "Turkey": "Türkiye",
    "The former Yugoslav Republic of Macedonia": "North Macedonia",
    "United Kingdom":
        "United Kingdom of Great Britain and Northern Ireland",
}

# Expected M49 codes for these names.
expected_codes = {
    "State of Palestine": "275",
    "Netherlands (Kingdom of the)": "528",
    "Türkiye": "792",
    "North Macedonia": "807",
    "United Kingdom of Great Britain and Northern Ireland": "826",
}

# Review only the rows that were unmatched.
review = still_unmatched.copy()

review["m49_code"] = (
    review["m49_code"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(3)
)

review["suggested_official_name"] = review["country"].map(name_mapping)

review["expected_code"] = review["suggested_official_name"].map(
    expected_codes
)

review["review_result"] = "MANUAL REVIEW REQUIRED"

# Name aliases whose codes agree with the current official reference.
valid_alias = (
    review["suggested_official_name"].notna()
    & (review["m49_code"] == review["expected_code"])
)

review.loc[valid_alias, "review_result"] = (
    "CODE MATCHES - NAME ALIAS"
)

# Taiwan entry requires checking the dataset's original source.
taiwan_mask = review["country"].eq("China,Taiwan")

review.loc[taiwan_mask, "suggested_official_name"] = pd.NA
review.loc[taiwan_mask, "expected_code"] = pd.NA
review.loc[taiwan_mask, "review_result"] = (
    "VERIFY AGAINST ORIGINAL DATA SOURCE"
)

display(
    review[
        [
            "country",
            "m49_code",
            "suggested_official_name",
            "expected_code",
            "review_result",
        ]
    ]
)


,country,m49_code,suggested_official_name,expected_code,review_result
19,"China,Taiwan",158,NaN,NaN,VERIFY AGAINST ORIGINAL DATA SOURCE
42,Palestine,275,State of Palestine,275,CODE MATCHES - NAME ALIAS
98,Netherlands,528,Netherlands (Kingdom of the),528,CODE MATCHES - NAME ALIAS
145,Turkey,792,Türkiye,792,CODE MATCHES - NAME ALIAS
149,The former Yugoslav Republic of Macedonia,807,North Macedonia,807,CODE MATCHES - NAME ALIAS
152,United Kingdom,826,United Kingdom of Great Britain and Northern I...,826,CODE MATCHES - NAME ALIAS


#### **Check country-code consistency**

In [183]:

import pandas as pd
import requests
from io import StringIO

# =========================================================
# 1. Three reference links
# =========================================================

links = {
    "UN_M49_Overview":
        "https://unstats.un.org/unsd/methodology/m49/overview/",

    "UN_M49_Methodology":
        "https://unstats.un.org/unsd/methodology/m49/",

    "GitHub_ISO_Regional_Codes":
        "https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv",
}

# =========================================================
# 2. Helper functions
# =========================================================

def normalize_name(series):
    return (
        series.astype("string")
        .str.strip()
        .str.casefold()
        .str.replace(r"\s+", " ", regex=True)
    )

def normalize_code(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(3)
    )

def download(url):
    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()
    return response

# =========================================================
# 3. Official UN M49 Overview
# =========================================================

response = download(links["UN_M49_Overview"])
tables = pd.read_html(StringIO(response.text))

un_table = next(
    table for table in tables
    if "Country or Area" in table.columns
    and "M49 Code" in table.columns
)

# Country/area codes
country_ref = pd.DataFrame({
    "name": un_table["Country or Area"],
    "code": un_table["M49 Code"],
    "record_type": "country_or_area",
    "source": "UN M49 Overview",
})

# Geographic region codes: match each name to its own code column
region_frames = []

region_columns = [
    ("Global Name", "Global Code", "global"),
    ("Region Name", "Region Code", "region"),
    ("Sub-region Name", "Sub-region Code", "sub_region"),
    (
        "Intermediate Region Name",
        "Intermediate Region Code",
        "intermediate_region",
    ),
]

for name_col, code_col, record_type in region_columns:
    if name_col in un_table.columns and code_col in un_table.columns:
        region_frames.append(pd.DataFrame({
            "name": un_table[name_col],
            "code": un_table[code_col],
            "record_type": record_type,
            "source": "UN M49 Overview",
        }))

un_regions = pd.concat(region_frames, ignore_index=True)

# =========================================================
# 4. Download the UN methodology page
# =========================================================

methodology_response = download(links["UN_M49_Methodology"])
methodology_tables = pd.read_html(
    StringIO(methodology_response.text)
)

methodology_frames = []

for table in methodology_tables:
    table = table.copy()
    table.columns = [str(col).strip() for col in table.columns]

    if "Country or Area" in table.columns:
        code_col = next(
            (
                col for col in table.columns
                if str(col).strip().casefold() == "m49 code"
            ),
            None
        )

        if code_col is not None:
            methodology_frames.append(pd.DataFrame({
                "name": table["Country or Area"],
                "code": table[code_col],
                "record_type": "country_or_area",
                "source": "UN M49 Methodology",
            }))

un_methodology = (
    pd.concat(methodology_frames, ignore_index=True)
    if methodology_frames
    else pd.DataFrame(
        columns=["name", "code", "record_type", "source"]
    )
)

# =========================================================
# 5. Download GitHub reference (ISO, not M49)
# =========================================================

github_response = download(links["GitHub_ISO_Regional_Codes"])
github = pd.read_csv(StringIO(github_response.text), dtype=str)

github_ref = pd.DataFrame({
    "name": github["name"],
    "code": github["country-code"],
    "record_type": "ISO_country_code",
    "source": "GitHub ISO reference",
})

# =========================================================
# 6. Build separate reference tables
# =========================================================

def prepare_reference(frame):
    frame = frame.copy()
    frame["name_key"] = normalize_name(frame["name"])
    frame["code_key"] = normalize_code(frame["code"])

    return frame.dropna(
        subset=["name_key", "code_key"]
    ).drop_duplicates(
        subset=["name_key", "code_key", "record_type", "source"]
    )

official_countries = prepare_reference(
    pd.concat(
        [country_ref, un_methodology],
        ignore_index=True
    )
)

official_regions = prepare_reference(un_regions)
github_ref = prepare_reference(github_ref)

# =========================================================
# 7. Prepare dataset
# =========================================================

data = world_food_loss[["country", "m49_code"]].copy()
data["name_key"] = normalize_name(data["country"])
data["actual_code"] = normalize_code(data["m49_code"])

# =========================================================
# 8. Validate country/area pairs against official M49
# =========================================================

valid_country_pairs = (
    official_countries[["name_key", "code_key"]]
    .drop_duplicates()
    .rename(columns={"code_key": "expected_code"})
)

checked = data.merge(
    valid_country_pairs.assign(pair_found=True),
    left_on=["name_key", "actual_code"],
    right_on=["name_key", "expected_code"],
    how="left",
    validate="many_to_one",
)

checked["pair_found"] = checked["pair_found"].eq(True)

known_country_names = set(official_countries["name_key"].dropna())
known_region_names = set(official_regions["name_key"].dropna())

checked["country_name_found"] = checked["name_key"].isin(
    known_country_names
)
checked["region_name_found"] = checked["name_key"].isin(
    known_region_names
)

checked["status"] = "NAME NOT FOUND - REVIEW"

checked.loc[
    checked["country_name_found"],
    "status"
] = "COUNTRY NAME FOUND - CHECK CODE"

checked.loc[
    checked["pair_found"],
    "status"
] = "VALID COUNTRY/AREA NAME + M49 CODE"

# Check geographic aggregate pairs separately
valid_region_pairs = (
    official_regions[["name_key", "code_key"]]
    .drop_duplicates()
)

region_pair_index = set(
    zip(valid_region_pairs["name_key"], valid_region_pairs["code_key"])
)

is_region_pair = pd.Series(
    [
        (name, code) in region_pair_index
        for name, code in zip(
            checked["name_key"], checked["actual_code"]
        )
    ],
    index=checked.index,
)

checked.loc[
    is_region_pair,
    "status"
] = "VALID GEOGRAPHIC REGION CODE"

checked.loc[
    checked["region_name_found"]
    & ~is_region_pair
    & ~checked["pair_found"],
    "status"
] = "REGION NAME FOUND - CHECK REGION CODE"

# =========================================================
# 9. Display report
# =========================================================

report = checked[
    [
        "country",
        "m49_code",
        "actual_code",
        "country_name_found",
        "region_name_found",
        "pair_found",
        "status",
    ]
].drop_duplicates()

display(report)

print("\nStatus summary:")
display(
    report["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="number_of_pairs")
)


# =========================================================
# 10. Show only unresolved issues
# =========================================================

issues = report[
    ~report["status"].str.startswith("VALID", na=False)
].copy()

print("\nRows requiring review:")
display(issues)

# Save issues to CSV
issues.to_csv(
    "../report/m49/m49_country_code_issues.csv",
    index=False
)

print(f"\nNumber of rows requiring review: {len(issues)}")
print("Issues report saved successfully.")


,country,m49_code,actual_code,country_name_found,region_name_found,pair_found,status
0,World,1,001,True,True,True,VALID GEOGRAPHIC REGION CODE
29,Myanmar,104,104,True,False,True,VALID COUNTRY/AREA NAME + M49 CODE
42,Burundi,108,108,True,False,True,VALID COUNTRY/AREA NAME + M49 CODE
551,Western Africa,11,011,True,True,True,VALID GEOGRAPHIC REGION CODE
555,Belarus,112,112,True,False,True,VALID COUNTRY/AREA NAME + M49 CODE
...,...,...,...,...,...,...,...
27857,United States of America,840,840,True,False,True,VALID COUNTRY/AREA NAME + M49 CODE
29573,Burkina Faso,854,854,True,False,True,VALID COUNTRY/AREA NAME + M49 CODE
30340,Uzbekistan,860,860,True,False,True,VALID COUNTRY/AREA NAME + M49 CODE
30352,Venezuela (Bolivarian Republic of),862,862,True,False,True,VALID COUNTRY/AREA NAME + M49 CODE



Status summary:


,status,number_of_pairs
0,VALID COUNTRY/AREA NAME + M49 CODE,137
1,VALID GEOGRAPHIC REGION CODE,14
2,NAME NOT FOUND - REVIEW,8



Rows requiring review:


,country,m49_code,actual_code,country_name_found,region_name_found,pair_found,status
2215,"Taiwan, Province of China",158,158,False,False,False,NAME NOT FOUND - REVIEW
2217,"China,Taiwan",158,158,False,False,False,NAME NOT FOUND - REVIEW
6891,Palestine,275,275,False,False,False,NAME NOT FOUND - REVIEW
10879,Côte d'Ivoire,384,384,False,False,False,NAME NOT FOUND - REVIEW
17946,Netherlands,528,528,False,False,False,NAME NOT FOUND - REVIEW
25890,Turkey,792,792,False,False,False,NAME NOT FOUND - REVIEW
26887,The former Yugoslav Republic of Macedonia,807,807,False,False,False,NAME NOT FOUND - REVIEW
27086,United Kingdom,826,826,False,False,False,NAME NOT FOUND - REVIEW



Number of rows requiring review: 8
Issues report saved successfully.


### **Check on column Country**

#### **Show missing values**

In [184]:
import re
import unicodedata
from difflib import get_close_matches

countries = world_food_loss["country"].copy()



def normalize_name(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", value.strip().casefold())

In [185]:
missing_countries = world_food_loss[
    world_food_loss["country"].isna()
]

print("Missing country values:", len(missing_countries))
display(missing_countries)

Missing country values: 0


,m49_code,country,region,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,food_supply_stage,treatment,cause_of_loss,sample_size,method_data_collection,reference,url,notes


#### **Show diff names for same country**

In [186]:
import pandas as pd
import unicodedata
import re

# Prepare the official UN M49 reference
un_reference = un_table[["Country or Area", "M49 Code"]].copy()

un_reference["m49_code"] = pd.to_numeric(
    un_reference["M49 Code"].astype(str).str.strip(),
    errors="coerce"
)

un_reference = un_reference.dropna(subset=["m49_code"])
un_reference["m49_code"] = un_reference["m49_code"].astype("int64")
un_reference["official_name"] = un_reference["Country or Area"].astype(str)
un_reference["normalized_official_name"] = (
    un_reference["official_name"].map(normalize_name)
)

# Keep all official names for each M49 code
official_names_by_code = (
    un_reference.groupby("m49_code")["official_name"]
    .apply(lambda names: sorted(set(names)))
    .to_dict()
)

# Prepare dataset country names
dataset_names = (
    world_food_loss[["country", "m49_code"]]
    .dropna(subset=["country", "m49_code"])
    .drop_duplicates()
    .copy()
)

dataset_names["m49_code"] = pd.to_numeric(
    dataset_names["m49_code"], errors="coerce"
)

dataset_names = dataset_names.dropna(subset=["m49_code"])
dataset_names["m49_code"] = dataset_names["m49_code"].astype("int64")
dataset_names["normalized_country"] = (
    dataset_names["country"].map(normalize_name)
)

# Match each dataset name against official names sharing its M49 code
dataset_names["official_names"] = dataset_names["m49_code"].map(
    official_names_by_code
)

dataset_names["official_names"] = dataset_names["official_names"].apply(
    lambda names: names if isinstance(names, list) else []
)

dataset_names["status"] = dataset_names.apply(
    lambda row: (
        "No official M49 code match"
        if not row["official_names"]
        else (
            "Exact name match"
            if row["normalized_country"] in [
                normalize_name(name) for name in row["official_names"]
            ]
            else "Name differs from official reference"
        )
    ),
    axis=1
)

# Display rows that need review
results = dataset_names[
    dataset_names["status"] != "Exact name match"
].copy()

results = results.rename(columns={
    "country": "dataset_country"
})

display(
    results[
        ["dataset_country", "m49_code", "official_names", "status"]
    ].sort_values(["status", "m49_code", "dataset_country"])
)

,dataset_country,m49_code,official_names,status
6891,Palestine,275,[State of Palestine],Name differs from official reference
10879,Côte d'Ivoire,384,[Côte d’Ivoire],Name differs from official reference
17946,Netherlands,528,[Netherlands (Kingdom of the)],Name differs from official reference
25890,Turkey,792,[Türkiye],Name differs from official reference
26887,The former Yugoslav Republic of Macedonia,807,[North Macedonia],Name differs from official reference
27086,United Kingdom,826,[United Kingdom of Great Britain and Northern ...,Name differs from official reference
0,World,1,[],No official M49 code match
2750,Africa,2,[],No official M49 code match
551,Western Africa,11,[],No official M49 code match
1939,Northern Africa,15,[],No official M49 code match


#### **Check on diff upper and lower names**

In [187]:
case_issues = world_food_loss[
    world_food_loss["country"].notna()
    & world_food_loss["country"].ne(
        world_food_loss["country"].str.title()
    )
][["country", "m49_code"]].drop_duplicates()

print("Names with non-standard capitalization:", len(case_issues))
display(case_issues)

Names with non-standard capitalization: 17


,country,m49_code
2215,"Taiwan, Province of China",158
2223,Democratic Republic of the Congo,180
10688,Iran (Islamic Republic of),364
10879,Côte d'Ivoire,384
12998,Democratic People's Republic of Korea,408
13002,Republic of Korea,410
13052,Lao People's Democratic Republic,418
13058,Latin America and the Caribbean,419
18013,Australia and New Zealand,53
21411,Saint Kitts and Nevis,659


#### **Check on spaces** 

In [188]:
country_text = world_food_loss["country"].astype("string")

outer_spaces = world_food_loss[
    country_text.notna()
    & country_text.ne(country_text.str.strip())
][["country", "m49_code"]]

internal_spaces = world_food_loss[
    country_text.notna()
    & country_text.str.contains(r"\s{2,}", regex=True, na=False)
][["country", "m49_code"]]

print("Leading/trailing spaces:", len(outer_spaces))
display(outer_spaces.drop_duplicates())

print("Repeated internal spaces:", len(internal_spaces))
display(internal_spaces.drop_duplicates())

Leading/trailing spaces: 0


,country,m49_code


Repeated internal spaces: 0


,country,m49_code


#### **Discovering mixed geographic regions with countries**

In [189]:
region_columns = [
    ("Global Name", "Global Code"),
    ("Region Name", "Region Code"),
    ("Sub-region Name", "Sub-region Code"),
    ("Intermediate Region Name", "Intermediate Region Code"),
]

region_frames = []

for name_col, code_col in region_columns:
    if name_col in un_table.columns and code_col in un_table.columns:
        temp = un_table[[name_col, code_col]].dropna().copy()
        temp.columns = ["region_name", "region_code"]
        region_frames.append(temp)

regions = pd.concat(region_frames, ignore_index=True).drop_duplicates()

region_names = set(
    regions["region_name"].map(normalize_name)
)

geographic_regions = world_food_loss[
    world_food_loss["country"].notna()
    & world_food_loss["country"].map(normalize_name).isin(region_names)
][["country", "m49_code"]].drop_duplicates()

print("Geographic region names found in country:",
      len(geographic_regions))

display(geographic_regions)

Geographic region names found in country: 14


,country,m49_code
0,World,1
551,Western Africa,11
1292,Asia,142
1293,Central Asia,143
1413,Western Asia,145
1939,Northern Africa,15
1944,Europe,150
2750,Africa,2
2754,Sub-Saharan Africa,202
3562,Northern America,21


### **Check on region**

#### **Show missins values**

In [190]:

def normalize_region(value):
    if pd.isna(value):
        return None

    value = unicodedata.normalize("NFKC", str(value))
    value = re.sub(r"\s+", " ", value.strip())
    return value.casefold()

df_region = world_food_loss.copy()

df_region["_region_normalized"] = (
    df_region["region"].map(normalize_region)
)

df_region["_country_normalized"] = (
    df_region["country"].map(normalize_region)
)

In [191]:
missing_region_report = (
    df_region.groupby(
        ["m49_code", "country"],
        dropna=False
    )
    .agg(
        total_rows=("region", "size"),
        missing_regions=("region", lambda s: s.isna().sum())
    )
    .reset_index()
)

missing_region_report["missing_percentage"] = (
    missing_region_report["missing_regions"]
    / missing_region_report["total_rows"]
    * 100
).round(2)

display(
    missing_region_report[
        missing_region_report["missing_regions"] > 0
    ].sort_values("missing_percentage", ascending=False)
)

,m49_code,country,total_rows,missing_regions,missing_percentage
0,1,World,29,29,100.00
1,2,Africa,4,4,100.00
2,4,Afghanistan,30,30,100.00
3,8,Albania,2,2,100.00
4,11,Western Africa,4,4,100.00
...,...,...,...,...,...
22,76,Brazil,43,17,39.53
99,504,Morocco,47,12,25.53
119,626,Timor-Leste,36,9,25.00
81,410,Republic of Korea,40,7,17.50


#### **Inconsistent Region Names and Different Regions per Country**

In [192]:
# Point 2: Inconsistent Region Names + Different Regions per Country

region_name_variants = (
    df_region.dropna(subset=["region"])
    .groupby("_region_normalized")
    .agg(
        original_names=(
            "region",
            lambda s: sorted(s.astype(str).unique().tolist())
        ),
        occurrences=("region", "size")
    )
    .reset_index()
)

region_name_variants["number_of_variants"] = (
    region_name_variants["original_names"].str.len()
)

# Get all distinct regions recorded for each country
country_regions = (
    df_region.dropna(subset=["country", "region"])
    .groupby(["m49_code", "country"])
    .agg(
        different_regions=(
            "region",
            lambda s: sorted(s.astype(str).unique().tolist())
        ),
        different_regions_count=("region", "nunique")
    )
    .reset_index()
)

# Identify region names that differ only in formatting/capitalization
inconsistent_names = region_name_variants[
    region_name_variants["number_of_variants"] > 1
].copy()

# Link those names to countries using the normalized region
variant_country_details = (
    df_region.dropna(subset=["country", "region"])
    .merge(
        inconsistent_names[
            ["_region_normalized", "original_names", "number_of_variants"]
        ],
        on="_region_normalized",
        how="inner"
    )
    .groupby(["m49_code", "country"])
    .agg(
        inconsistent_region_names=(
            "original_names",
            lambda groups: sorted({
                name
                for group in groups
                for name in group
            })
        ),
        inconsistent_region_count=(
            "_region_normalized", "nunique"
        )
    )
    .reset_index()
)

# Combine both reports
point_2_report = country_regions.merge(
    variant_country_details,
    on=["m49_code", "country"],
    how="left"
)

display(
    point_2_report[
        [
            "m49_code",
            "country",
            "different_regions_count",
            "different_regions",
            "inconsistent_region_count",
            "inconsistent_region_names"
        ]
    ].sort_values(
        ["inconsistent_region_count", "country"],
        ascending=[False, True],
        na_position="last"
    )
)

,m49_code,country,different_regions_count,different_regions,inconsistent_region_count,inconsistent_region_names
9,231,Ethiopia,23,[Adami Tulu - Jido Kombolcha and Adama Distric...,2.0,"[Amhara Region, Amhara region, Southern Natio..."
12,288,Ghana,34,[? Sites: coastal and transition ecological zo...,2.0,"[Bawku, Wa Tamale, Yendi, and Techiman, Bawku..."
6,156,China,38,"[Anhui, Beijing, Beijing, Xinfadi and Jinxiuda...",1.0,"[Xiasha Town, Xiasha town]"
0,50,Bangladesh,32,[4 Upazillas of Chapai Nowabgonj and 2 Upazill...,NaN,NaN
7,204,Benin,1,[Parakou (Borgou)],NaN,NaN
1,76,Brazil,3,"[Bahia, Rio Grande do Sul, State of Bahia, Sla...",NaN,NaN
2,116,Cambodia,1,"[Kandal Province, and wholesalers and retailer...",NaN,NaN
3,120,Cameroon,2,"[Boyo, Yaoundé]",NaN,NaN
5,152,Chile,1,[North Patagonia Region],NaN,NaN
8,218,Ecuador,3,"[El Carchi, El Carchi, Imbabura, and Pichincha...",NaN,NaN


#### **Extra Spaces**

In [193]:
region_text = df_region["region"].astype("string")

extra_spaces = df_region.loc[
    region_text.notna()
    & (
        region_text.ne(region_text.str.strip())
        | region_text.str.contains(r"\s{2,}", regex=True, na=False)
    ),
    ["m49_code", "country", "region"]
].drop_duplicates()

display(extra_spaces)

print("Rows with extra spaces:", len(extra_spaces))

,m49_code,country,region
2141,156,China,Dingxi Prefecture in Gansu
3682,231,Ethiopia,"Southern Nations, Nationalities and Peoples"
3687,231,Ethiopia,"Southern Nations, Nationalities and Peoples"
3690,231,Ethiopia,"Southern Nations, Nationalities and Peoples"
7657,288,Ghana,coastal and transition ecological zones of Ghana
7809,288,Ghana,"Bawku, Wa Tamale, Yendi , Techiman, and Kintampo"
7872,288,Ghana,"Bawku, Wa Tamale, Yendi, and Techiman"
7887,288,Ghana,"Bawku, Wa Tamale, Yendi and Techiman"
12613,404,Kenya,"Upper Athi River Region; Kilala, Ikanganya, K..."
22616,704,Viet Nam,"An Giang, Sóc Trăng, Bạc Liêu and Kiên Giang ..."


Rows with extra spaces: 12


#### **Same Region Used Across Different Countries**

In [194]:
shared_regions = (
    df_region.dropna(
        subset=["_region_normalized", "_country_normalized"]
    )
    .groupby("_region_normalized")
    .agg(
        region_names=(
            "region",
            lambda s: sorted(s.astype(str).unique().tolist())
        ),
        countries=(
            "country",
            lambda s: sorted(s.astype(str).unique().tolist())
        ),
        country_count=("_country_normalized", "nunique"),
        records=("country", "size")
    )
    .reset_index()
)

display(
    shared_regions[
        shared_regions["country_count"] > 1
    ].sort_values("country_count", ascending=False)
)

,_region_normalized,region_names,countries,country_count,records
275,punjab,[Punjab],"[Bangladesh, India, Pakistan]",3,6
21,assam,[Assam],"[India, Peru]",2,5
171,karnataka,[Karnataka],"[Bangladesh, India]",2,13
246,national average,[National average],"[Egypt, Jordan]",2,12
350,uttar pradesh,[Uttar Pradesh],"[Bangladesh, India]",2,3


#### **Mixed Geographic Levels**

In [195]:
region_level_review = (
    df_region.dropna(subset=["region"])
    .groupby(["region"])
    .agg(
        records=("region", "size"),
        country_count=("country", "nunique"),
        countries=(
            "country",
            lambda s: sorted(s.dropna().astype(str).unique().tolist())
        ),
        m49_codes=(
            "m49_code",
            lambda s: sorted(s.dropna().astype(str).unique().tolist())
        )
    )
    .reset_index()
    .sort_values("region")
)

display(region_level_review)

,region,records,country_count,countries,m49_codes
0,4 Upazillas of Chapai Nowabgonj and 2 Upazilla...,2,1,[Bangladesh],[50]
1,? Sites: coastal and transition ecological zon...,1,1,[Ghana],[288]
2,Adami Tulu - Jido Kombolcha and Adama Districts,3,1,[Ethiopia],[231]
3,Addis Ababa and Adama cities,2,1,[Ethiopia],[231]
4,"Ait Baamrane, Rhamna",1,1,[Morocco],[504]
...,...,...,...,...,...
373,coastal and transition ecological zones of Ghana,7,1,[Ghana],[288]
374,forest and coastal ecological zones of Ghana,13,1,[Ghana],[288]
375,"four main potatogrowing counties in Kenya, nam...",4,1,[Kenya],[404]
376,from Jumla,1,1,[Nepal],[524]


#### **Duplicate or Equivalent Names**

In [196]:
equivalent_region_names = (
    df_region.dropna(subset=["region"])
    .groupby("_region_normalized")
    .agg(
        original_names=(
            "region",
            lambda s: sorted(s.astype(str).unique().tolist())
        ),
        number_of_names=("region", "nunique"),
        total_records=("region", "size")
    )
    .reset_index()
)

display(
    equivalent_region_names[
        equivalent_region_names["number_of_names"] > 1
    ]
)

,_region_normalized,original_names,number_of_names,total_records
8,amhara region,"[Amhara Region, Amhara region]",2,4
32,"bawku, wa tamale, yendi, and techiman","[Bawku, Wa Tamale, Yendi, and Techiman, Bawku...",2,15
77,coastal and transition ecological zones of ghana,[coastal and transition ecological zones of Gh...,2,8
324,"southern nations, nationalities and peoples","[Southern Nations, Nationalities and People...",3,8
365,xiasha town,"[Xiasha Town, Xiasha town]",2,7


#### **Validate Missing Regions**

In [197]:
missing_region_records = df_region.loc[
    df_region["region"].isna(),
    [
        "m49_code",
        "country",
        "region",
        "commodity",
        "year",
        "food_supply_stage",
        "activity",
        "cause_of_loss"
    ]
].copy()

display(missing_region_records)

print("Total records with missing region:",
      len(missing_region_records))

print("Countries with missing region:",
      missing_region_records["country"].nunique())

print("\nMissing region records by country:")
display(
    missing_region_records.groupby(
        ["m49_code", "country"],
        dropna=False
    )
    .size()
    .reset_index(name="missing_region_records")
    .sort_values("missing_region_records", ascending=False)
)

,m49_code,country,region,commodity,year,food_supply_stage,activity,cause_of_loss
0,1,World,NaN,Wheat,2022,Processing,Processing,NaN
1,1,World,NaN,Maize (corn),2022,Processing,Processing,NaN
2,1,World,NaN,Maize (corn),2022,Processing,Processing,NaN
3,1,World,NaN,Maize (corn),2022,Processing,Processing,NaN
4,1,World,NaN,Maize (corn),2022,Processing,Processing,NaN
...,...,...,...,...,...,...,...,...
31460,894,Zambia,NaN,Millet,2000,Storage,Storage,NaN
31461,894,Zambia,NaN,Millet,2000,NaN,"Shelling, Threshing",NaN
31462,894,Zambia,NaN,Millet,2000,Farm,Transportation,NaN
31463,894,Zambia,NaN,Millet,2000,Transport,Transportation,NaN


Total records with missing region: 29429
Countries with missing region: 157

Missing region records by country:


,m49_code,country,missing_region_records
152,840,United States of America,1715
68,356,India,1225
49,231,Ethiopia,1072
79,404,Kenya,969
145,800,Uganda,848
...,...,...,...
55,266,Gabon,1
42,191,Croatia,1
37,158,"China,Taiwan",1
122,659,Saint Kitts and Nevis,1


#### **Incorrect Country–Region Mapping**

In [198]:
import pandas as pd
import requests
from io import StringIO
from IPython.display import display

# All reference links
reference_links = {
    "un_m49_official_page": (
        "https://unstats.un.org/unsd/methodology/m49/overview/"
    ),
    "un_m49_csv": (
        "https://raw.githubusercontent.com/omnika-datastore/"
        "unsd-m49-standard-area-codes/main/"
        "2022-09-24__CSV_UNSD_M49.csv"
    ),
    "lukes_iso_m49_csv": (
        "https://raw.githubusercontent.com/lukes/"
        "ISO-3166-Countries-with-Regional-Codes/master/all/all.csv"
    ),
    "mledoze_countries_json": (
        "https://raw.githubusercontent.com/mledoze/"
        "countries/master/countries.json"
    ),
    "datahub_country_codes_csv": (
        "https://datahub.io/core/country-codes/r/country-codes.csv"
    ),
    "geojson_countries": (
        "https://raw.githubusercontent.com/datasets/"
        "geo-countries/master/data/countries.geojson"
    ),
}

references = {}
load_results = []

for name, url in reference_links.items():
    try:
        if name == "un_m49_official_page":
            # Official UN page: extract its HTML tables
            tables = pd.read_html(url)

            references[name] = tables

            load_results.append({
                "source": name,
                "status": "Loaded",
                "rows": sum(len(table) for table in tables),
                "columns_or_keys": str([
                    table.columns.tolist() for table in tables
                ]),
                "url": url
            })
            continue

        response = requests.get(url, timeout=30)
        response.raise_for_status()

        if name.endswith("_csv"):
            data = pd.read_csv(
                StringIO(response.text),
                dtype="string"
            )

        elif name.endswith("_json"):
            data = pd.json_normalize(response.json())

        elif name == "geojson_countries":
            geojson = response.json()
            data = pd.json_normalize(geojson["features"])

        else:
            data = response.json()

        references[name] = data

        load_results.append({
            "source": name,
            "status": "Loaded",
            "rows": len(data) if isinstance(data, pd.DataFrame) else None,
            "columns_or_keys": str(
                data.columns.tolist()
                if isinstance(data, pd.DataFrame)
                else list(data.keys())
            ),
            "url": url
        })

    except Exception as e:
        load_results.append({
            "source": name,
            "status": f"Failed: {type(e).__name__}: {e}",
            "rows": None,
            "columns_or_keys": None,
            "url": url
        })

load_report = pd.DataFrame(load_results)

display(load_report)

,source,status,rows,columns_or_keys,url
0,un_m49_official_page,Loaded,1488.0,"[['Global Code', 'Global Name', 'Region Code',...",https://unstats.un.org/unsd/methodology/m49/ov...
1,un_m49_csv,Failed: ParserError: Error tokenizing data. C ...,NaN,NaN,https://raw.githubusercontent.com/omnika-datas...
2,lukes_iso_m49_csv,Loaded,249.0,"['name', 'alpha-2', 'alpha-3', 'country-code',...",https://raw.githubusercontent.com/lukes/ISO-31...
3,mledoze_countries_json,Loaded,250.0,"['tld', 'cca2', 'ccn3', 'cca3', 'cioc', 'indep...",https://raw.githubusercontent.com/mledoze/coun...
4,datahub_country_codes_csv,Loaded,249.0,"['FIFA', 'Dial', 'ISO3166-1-Alpha-3', 'MARC', ...",https://datahub.io/core/country-codes/r/countr...
5,geojson_countries,Loaded,258.0,"['type', 'properties.name', 'properties.ISO316...",https://raw.githubusercontent.com/datasets/geo...


In [199]:
import pandas as pd
import re
import unicodedata

def normalize_name(value):
    if pd.isna(value):
        return ""

    value = unicodedata.normalize("NFKC", str(value))
    value = value.strip().casefold()
    value = re.sub(r"\s+", " ", value)

    return value


# Get the official UN M49 table
un_tables = references["un_m49_official_page"]

un_m49 = next(
    table.copy()
    for table in un_tables
    if {
        "Country or Area",
        "M49 Code",
        "Region Name"
    }.issubset(table.columns)
)

un_m49["country_key"] = un_m49["Country or Area"].map(
    normalize_name
)

un_m49["m49_code"] = (
    un_m49["M49 Code"]
    .astype("string")
    .str.strip()
    .str.zfill(3)
)

un_m49["official_region"] = un_m49["Region Name"].astype("string").str.strip()

display(
    un_m49[
        ["Country or Area", "m49_code", "official_region"]
    ].head()
)

,Country or Area,m49_code,official_region
0,Algeria,012,Africa
1,Egypt,818,Africa
2,Libya,434,Africa
3,Morocco,504,Africa
4,Sudan,729,Africa


In [200]:
# Validate country + M49 code as a pair, then inspect the region label.
# IMPORTANT: FAO's "region" can be a source-specific/local region (e.g. Upper Egypt),
# so a difference from UN's top-level Region Name is NOT automatically an error.

food = world_food_loss.copy()

def normalize_m49_code(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.endswith(".0") and text[:-2].isdigit():
        text = text[:-2]
    if text.isdigit() and len(text) <= 3:
        return text.zfill(3)
    return text

food["country_key"] = food["country"].map(normalize_name)
food["m49_code_key"] = food["m49_code"].map(normalize_m49_code)
food["region_key"] = food["region"].map(normalize_name)

# Country/area reference from official UN M49 table.
country_lookup = (
    un_m49[["country_key", "m49_code", "Country or Area", "official_region"]]
    .drop_duplicates()
    .rename(columns={
        "m49_code": "official_m49_code",
        "Country or Area": "official_country_name",
        "official_region": "un_region"
    })
)

checked = food.merge(
    country_lookup,
    on="country_key",
    how="left",
    validate="many_to_one"
)

checked["country_code_status"] = "PASS"
checked.loc[
    checked["country_key"].eq("") | checked["m49_code_key"].eq(""),
    "country_code_status"
] = "MISSING_COUNTRY_OR_CODE"
checked.loc[
    checked["country_key"].ne("")
    & checked["m49_code_key"].ne("")
    & checked["official_m49_code"].isna(),
    "country_code_status"
] = "COUNTRY_NAME_NOT_FOUND_IN_UN_M49"
checked.loc[
    checked["official_m49_code"].notna()
    & checked["m49_code_key"].ne(
        checked["official_m49_code"].map(normalize_m49_code)
    ),
    "country_code_status"
] = "COUNTRY_CODE_MISMATCH"

# A difference in region labels is only a review candidate.
checked["region_review_status"] = "NOT_COMPARABLE"
comparable_region = (
    checked["region_key"].ne("")
    & checked["un_region"].notna()
    & checked["un_region"].astype("string").str.strip().ne("")
)
checked.loc[comparable_region, "region_review_status"] = "REVIEW_REGION_LEVEL"
checked.loc[
    comparable_region
    & checked["region_key"].eq(checked["un_region"].map(normalize_name)),
    "region_review_status"
] = "MATCHES_UN_TOP_LEVEL_REGION"

country_code_issues = checked.loc[
    checked["country_code_status"].ne("PASS"),
    [
        "country", "m49_code", "region", "official_country_name",
        "official_m49_code", "country_code_status", "commodity",
        "year", "reference", "url"
    ]
].drop_duplicates()

region_level_review = checked.loc[
    checked["region_review_status"].eq("REVIEW_REGION_LEVEL"),
    [
        "country", "m49_code", "region", "un_region",
        "region_review_status", "commodity", "year", "reference"
    ]
].drop_duplicates()

print("Country/code validation status:")
display(checked["country_code_status"].value_counts(dropna=False).rename_axis("status").reset_index(name="records"))
print("Country/code issues:", len(country_code_issues))
display(country_code_issues.head(100))

print("Region labels needing geographic-level review:", len(region_level_review))
display(region_level_review.head(100))

Country/code validation status:


,status,records
0,PASS,30735
1,COUNTRY_NAME_NOT_FOUND_IN_UN_M49,730


Country/code issues: 282


,country,m49_code,region,official_country_name,official_m49_code,country_code_status,commodity,year,reference,url
0,World,1,NaN,NaN,<NA>,COUNTRY_NAME_NOT_FOUND_IN_UN_M49,Wheat,2022,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN
1,World,1,NaN,NaN,<NA>,COUNTRY_NAME_NOT_FOUND_IN_UN_M49,Maize (corn),2022,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN
6,World,1,NaN,NaN,<NA>,COUNTRY_NAME_NOT_FOUND_IN_UN_M49,Rice,2022,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN
7,World,1,NaN,NaN,<NA>,COUNTRY_NAME_NOT_FOUND_IN_UN_M49,Sorghum,2022,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN
8,World,1,NaN,NaN,<NA>,COUNTRY_NAME_NOT_FOUND_IN_UN_M49,Barley,2022,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN
...,...,...,...,...,...,...,...,...,...,...
11094,Côte d'Ivoire,384,NaN,NaN,<NA>,COUNTRY_NAME_NOT_FOUND_IN_UN_M49,Rice,2010,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...
11099,Côte d'Ivoire,384,NaN,NaN,<NA>,COUNTRY_NAME_NOT_FOUND_IN_UN_M49,Sorghum,2010,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...
11103,Côte d'Ivoire,384,NaN,NaN,<NA>,COUNTRY_NAME_NOT_FOUND_IN_UN_M49,Millet,2010,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...
11108,Côte d'Ivoire,384,NaN,NaN,<NA>,COUNTRY_NAME_NOT_FOUND_IN_UN_M49,Maize (corn),2009,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...


Region labels needing geographic-level review: 666


,country,m49_code,region,un_region,region_review_status,commodity,year,reference
568,Cambodia,116,"Kandal Province, and wholesalers and retailers...",Asia,REVIEW_REGION_LEVEL,Tomatoes,2013,Buntong; B.; Srilaong; V.; Wasusri; T.; Kanlay...
647,Cameroon,120,Boyo,Africa,REVIEW_REGION_LEVEL,Bananas,2020,"Nkwain, K., Odiaka, E., Ikwuba, A.; Nkwi, G. E..."
697,Cameroon,120,Yaoundé,Africa,REVIEW_REGION_LEVEL,Watermelons,2017,Aristide G. Kamda Silapeux;Roger Ponka;Chiara ...
698,Cameroon,120,Yaoundé,Africa,REVIEW_REGION_LEVEL,Avocados,2017,Aristide G. Kamda Silapeux;Roger Ponka;Chiara ...
699,Cameroon,120,Yaoundé,Africa,REVIEW_REGION_LEVEL,Bananas,2017,Aristide G. Kamda Silapeux;Roger Ponka;Chiara ...
...,...,...,...,...,...,...,...,...
3312,Benin,204,Parakou (Borgou),Africa,REVIEW_REGION_LEVEL,"Mangoes, guavas and mangosteens",2005,Jean-François Vayssières; Sam Korie; Ousmane C...
3574,Ecuador,218,"El Carchi, Imbabura, and Pichincha",Americas,REVIEW_REGION_LEVEL,Potatoes,2016,Luciana Delgado; Monica Schuster-; Maximo Tore...
3575,Ecuador,218,Pichincha,Americas,REVIEW_REGION_LEVEL,Potatoes,2016,Luciana Delgado; Monica Schuster-; Maximo Tore...
3576,Ecuador,218,El Carchi,Americas,REVIEW_REGION_LEVEL,Potatoes,2016,Luciana Delgado; Monica Schuster-; Maximo Tore...


#### **Unexpected Region Values**

In [201]:
# Collect all official geographic labels by level from the UN M49 table.
region_specs = [
    ("Global Name", "Global Code", "global"),
    ("Region Name", "Region Code", "region"),
    ("Sub-region Name", "Sub-region Code", "sub_region"),
    ("Intermediate Region Name", "Intermediate Region Code", "intermediate_region"),
]

region_reference_frames = []
for name_col, code_col, level in region_specs:
    if name_col in un_m49.columns and code_col in un_m49.columns:
        part = un_m49[[name_col, code_col]].copy()
        part.columns = ["official_region_name", "official_region_code"]
        part["geographic_level"] = level
        part = part.dropna(subset=["official_region_name", "official_region_code"])
        part["official_region_name"] = part["official_region_name"].astype("string").str.strip()
        part["official_region_code"] = part["official_region_code"].map(normalize_m49_code)
        part = part[part["official_region_name"].ne("")]
        region_reference_frames.append(part)

un_geographic_reference = (
    pd.concat(region_reference_frames, ignore_index=True).drop_duplicates()
    if region_reference_frames else
    pd.DataFrame(columns=["official_region_name", "official_region_code", "geographic_level"])
)

un_geographic_reference["region_key"] = un_geographic_reference[
    "official_region_name"
].map(normalize_name)

valid_region_keys = set(un_geographic_reference["region_key"].dropna())

print("UN geographic labels by level:")
display(
    un_geographic_reference.groupby("geographic_level")
    .agg(unique_names=("region_key", "nunique"),
         unique_codes=("official_region_code", "nunique"))
    .reset_index()
)
display(un_geographic_reference.sort_values(["geographic_level", "official_region_name"]).head(100))

UN geographic labels by level:


,geographic_level,unique_names,unique_codes
0,global,1,1
1,intermediate_region,7,7
2,region,5,5
3,sub_region,17,17


,official_region_name,official_region_code,geographic_level,region_key
0,World,001,global,world
795,Caribbean,029,intermediate_region,caribbean
823,Central America,013,intermediate_region,central america
742,Eastern Africa,014,intermediate_region,eastern africa
764,Middle Africa,017,intermediate_region,middle africa
831,South America,005,intermediate_region,south america
773,Southern Africa,018,intermediate_region,southern africa
778,Western Africa,011,intermediate_region,western africa
248,Africa,002,region,africa
308,Americas,019,region,americas


In [202]:
# Identify observed region labels not present in UN M49's global/region/subregion hierarchy.
# These are review candidates, NOT necessarily errors: FAO may contain local or study-specific regions.
observed_regions = (
    world_food_loss["region"]
    .dropna()
    .astype("string")
    .str.strip()
    .drop_duplicates()
)

unexpected_regions = observed_regions[
    ~observed_regions.map(normalize_name).isin(valid_region_keys)
]

unexpected_region_report = (
    world_food_loss[
        world_food_loss["region"].map(normalize_name).isin(
            set(unexpected_regions.map(normalize_name))
        )
    ]
    .groupby("region", dropna=False)
    .agg(
        row_count=("region", "size"),
        country_count=("country", "nunique"),
        sample_countries=(
            "country",
            lambda values: ", ".join(values.dropna().astype(str).unique()[:5])
        ),
        sample_commodities=(
            "commodity",
            lambda values: ", ".join(values.dropna().astype(str).unique()[:5])
        )
    )
    .reset_index()
    .sort_values("row_count", ascending=False)
)

print("Observed region labels not in UN M49 hierarchy:", len(unexpected_regions))
print("Interpret these as source-specific labels to classify, not auto-delete.")
display(unexpected_region_report.head(200))

Observed region labels not in UN M49 hierarchy: 377
Interpret these as source-specific labels to classify, not auto-delete.


,region,row_count,country_count,sample_countries,sample_commodities
359,Western Cape Province,59,1,South Africa,"Pears, Grapes"
78,Comilla,45,1,Bangladesh,"Potatoes, Rice"
349,Uttarakhand,42,1,India,"Cabbages, Cauliflowers and broccoli, Tomatoes,..."
282,Rangpur and Dinazpur district,36,1,Bangladesh,"Wheat, Rice"
31,"Bawku, Wa Tamale, Yendi , Techiman, and Kintampo",36,1,Ghana,"Tomatoes, Yams"
...,...,...,...,...,...
306,Shanxi,3,1,China,"Wheat, Maize (corn), Rice"
253,Nubaria,3,1,Egypt,Grapes
51,Bogra and Dakha,3,1,Bangladesh,Potatoes
280,Rampura,3,1,India,"Wheat, Maize (corn)"


### **Check on cpc_code**

#### **Show missing values**

In [203]:

missing_cpc = df_region.loc[
    df_region["cpc_code"].isna(),
    ["country", "region", "commodity", "cpc_code"]
].copy()

display(missing_cpc)

print("Missing cpc_code records:", len(missing_cpc))
print(
    "Percentage missing:",
    round(df_region["cpc_code"].isna().mean() * 100, 2),
    "%"
)

,country,region,commodity,cpc_code


Missing cpc_code records: 0
Percentage missing: 0.0 %


#### **Show Code with diff syntax**

In [204]:
cpc_text = df_region["cpc_code"].astype("string")
cpc_trimmed = cpc_text.str.strip()

cpc_format_issues = df_region.loc[
    cpc_text.notna()
    & (
        cpc_text.ne(cpc_trimmed)
        | cpc_trimmed.str.contains(r"\s", regex=True, na=False)
    ),
    ["country", "region", "commodity", "cpc_code"]
].copy()

display(cpc_format_issues)

print("Records with whitespace issues:", len(cpc_format_issues))

,country,region,commodity,cpc_code


Records with whitespace issues: 0


#### **Discovering mixed geographic regions with countries**

***CPC 2.1***

In [205]:
cpc_clean = df_region["cpc_code"].astype("string").str.strip()

# Preliminary syntax check for CPC 2.1 codes.
# This does not validate codes against the official CPC 2.1 list.
expected_cpc_pattern = r"\d+(?:\.\d+)?"

invalid_cpc_format = df_region.loc[
    cpc_clean.notna()
    & cpc_clean.ne("")
    & ~cpc_clean.str.fullmatch(
        expected_cpc_pattern,
        na=False
    ),
    ["country", "region", "commodity", "cpc_code"]
].copy()

display(invalid_cpc_format)

print(
    "Records outside the preliminary format:",
    len(invalid_cpc_format)
)

,country,region,commodity,cpc_code
9893,India,NaN,"Meat of goat, fresh or chilled (indigenous)",21116i
19441,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19463,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19483,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19503,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19523,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19543,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f


Records outside the preliminary format: 7


#### **The compatibility between a CPC Code (Central Product Classification) and a Commodity Code (HS Code)**

In [206]:
# Official CPC 2.1 classification and CPC 2.1 Expanded for Agriculture references
cpc_reference_url = {
    "UNSD_CPC21_Official_Excel":
        "https://unstats.un.org/unsd/classifications/Econ/Download/In%20Text/CPC_Ver_2.1_Exp_Notes_Updated_3Apr2025.xlsx",

    "FAO_Caliper_Classifications_Download":
        "https://www.fao.org/statistics/caliper/tools/download/en",

    "FAO_FCL_to_CPC21_Expanded_Agriculture_Mapping":
        "https://stats.fao.org/caliper/download/FCL/FCL_CPC21AG_mappings.csv",

    "UNSD_CPC21_Code_01199_Details":
        "https://unstats.un.org/unsd/classifications/Econ/Detail/EN/1074/01199"
}

# Load the official CPC 2.1 Excel workbook
cpc_workbook = pd.read_excel(
    cpc_reference_url["UNSD_CPC21_Official_Excel"],
    sheet_name=None,
    dtype=str
)

print("Available sheets:")
for sheet_name, sheet_df in cpc_workbook.items():
    print(f"\nSheet: {sheet_name}")
    print("Columns:", sheet_df.columns.tolist())
    display(sheet_df.head(3))

Available sheets:

Sheet: CPC2.1
Columns: ['CPC Ver. 2.1 Code', 'CPC Ver. 2.1 Title', 'CPC Ver. 2.1 Explanatory Note - Inclusions', 'CPC Ver. 2.1 Explanatory Note - Exclusions']


,CPC Ver. 2.1 Code,CPC Ver. 2.1 Title,CPC Ver. 2.1 Explanatory Note - Inclusions,CPC Ver. 2.1 Explanatory Note - Exclusions
0,0,"Agriculture, forestry and fishery products",NaN,NaN
1,01,"Products of agriculture, horticulture and mark...",NaN,NaN
2,011,Cereals,"This group includes cereals, i.e. annual plant...",NaN



Sheet: 61_62
Columns: ['Four-digit and five-digit codes in Divisions 61 and 62', 'Unnamed: 1', 'Unnamed: 2']


,Four-digit and five-digit codes in Divisions 61 and 62,Unnamed: 1,Unnamed: 2
0,Notes:\n1. The codes in the table below has th...,NaN,NaN
1,CPC Ver. 2.1 Code,CPC Ver. 2.1 Title,CPC Ver. 2.1 Explanatory Note - Inclusions
2,***1,"Trade services, of raw agricultural products a...",NaN



Sheet: Notes
Columns: ['Note on Updates and Corrections to CPC Ver. 2.1:', 'Unnamed: 1']


,Note on Updates and Corrections to CPC Ver. 2.1:,Unnamed: 1
0,Last updated by UNSD on 3 April 2025.,NaN
1,This Excel document includes important updates...,NaN
2,NaN,NaN


In [207]:

# Load the official CPC 2.1 classification sheet
cpc_reference = cpc_workbook["CPC2.1"].copy()

code_col = "CPC Ver. 2.1 Code"
title_col = "CPC Ver. 2.1 Title"

# Preserve codes as strings and remove surrounding whitespace
cpc_reference["_cpc_code_clean"] = (
    cpc_reference[code_col].astype("string").str.strip()
)

cpc_reference["_cpc_title_clean"] = (
    cpc_reference[title_col].astype("string").str.strip()
)

# Remove rows without a usable code
cpc_reference = cpc_reference.loc[
    cpc_reference["_cpc_code_clean"].notna()
    & cpc_reference["_cpc_code_clean"].ne("")
].copy()

# Prepare the dataset codes
data_cpc = df_region["cpc_code"].astype("string").str.strip()

# Build a lookup set from official CPC 2.1 codes
official_codes = set(cpc_reference["_cpc_code_clean"])

# Find codes that are not present in the official reference
invalid_cpc_codes = df_region.loc[
    data_cpc.notna()
    & data_cpc.ne("")
    & ~data_cpc.isin(official_codes),
    ["country", "region", "commodity", "cpc_code"]
].copy()

display(invalid_cpc_codes)

print("Rows with codes not found in CPC 2.1:", len(invalid_cpc_codes))
print(
    "Distinct codes not found:",
    invalid_cpc_codes["cpc_code"].nunique()
)

,country,region,commodity,cpc_code
16,World,NaN,Cereals n.e.c.,01199.90
17,World,NaN,Cereals n.e.c.,01199.90
18,World,NaN,Cereals n.e.c.,01199.90
19,World,NaN,Pot barley,23140.04
20,World,NaN,Cereal preparations,23140.08
...,...,...,...,...
30691,Venezuela (Bolivarian Republic of),NaN,"Pig, Butcher Fat",21511.02
30693,Venezuela (Bolivarian Republic of),NaN,Cheese from Whole Cow Milk,22251.01
30695,Venezuela (Bolivarian Republic of),NaN,Flour of Maize,23120.03
30696,Venezuela (Bolivarian Republic of),NaN,"Rice, Milled",23161.02


Rows with codes not found in CPC 2.1: 1629
Distinct codes not found: 71


#### **Displaying each used code and comparing it with the official title**

In [208]:
# Aggregate commodities used with each CPC code
used_codes = (
    df_region.dropna(subset=["cpc_code"])
    .assign(
        _cpc_code_clean=lambda d: (
            d["cpc_code"].astype("string").str.strip()
        )
    )
    .groupby("_cpc_code_clean")
    .agg(
        commodities=(
            "commodity",
            lambda s: sorted(s.dropna().astype(str).unique().tolist())
        ),
        records=("commodity", "size")
    )
    .reset_index()
)

# Match dataset codes to the official CPC 2.1 reference
official_lookup = (
    cpc_reference[
        ["_cpc_code_clean", "_cpc_title_clean"]
    ]
    .drop_duplicates(subset=["_cpc_code_clean"])
)

cpc_code_review = used_codes.merge(
    official_lookup,
    on="_cpc_code_clean",
    how="left",
    indicator=True
)

cpc_code_review = cpc_code_review.rename(
    columns={
        "_cpc_code_clean": "cpc_code",
        "_cpc_title_clean": "official_cpc_title",
        "_merge": "reference_match"
    }
)

display(
    cpc_code_review.sort_values(
        ["reference_match", "cpc_code"]
    )
)

,cpc_code,commodities,records,official_cpc_title,reference_match
13,01199.02,[Mixed grain],57,<NA>,left_only
14,01199.90,"[Cereals n.e.c., Other cereals n.e.]",30,<NA>,left_only
28,01239.01,[Okra],91,<NA>,left_only
29,01241.90,"[Other beans, green]",78,<NA>,left_only
34,01253.01,"[Onions and shallots, green]",141,<NA>,left_only
...,...,...,...,...,...
181,23162,[Husked rice],20,Husked rice,both
188,23520,[Refined sugar],51,Refined sugar,both
189,23710,"[Uncooked pasta, not stuffed or otherwise prep...",9,"Uncooked pasta, not stuffed or otherwise prepared",both
190,23911,"[Coffee, decaffeinated or roasted]",2,"Coffee, decaffeinated or roasted",both


#### **Checking only the codes that are missing from the reference**

In [209]:
not_found = cpc_code_review.loc[
    cpc_code_review["reference_match"] == "left_only"
].copy()

display(not_found)

print("Distinct unmatched codes:", len(not_found))

,cpc_code,commodities,records,official_cpc_title,reference_match
13,01199.02,[Mixed grain],57,<NA>,left_only
14,01199.90,"[Cereals n.e.c., Other cereals n.e.]",30,<NA>,left_only
28,01239.01,[Okra],91,<NA>,left_only
29,01241.90,"[Other beans, green]",78,<NA>,left_only
34,01253.01,"[Onions and shallots, green]",141,<NA>,left_only
...,...,...,...,...,...
187,2351f,[Raw cane or beet sugar (centrifugal only)],6,<NA>,left_only
191,23993.03,"[Eggs, dried]",6,<NA>,left_only
192,24212.02,[Wine],1,<NA>,left_only
194,39120.04,[Bran of Maize],14,<NA>,left_only


Distinct unmatched codes: 71


#### **Reviewing suspicious codes**

In [210]:
display(
    cpc_code_review.loc[
        cpc_code_review["reference_match"] == "left_only",
        ["cpc_code", "commodities", "records"]
    ].sort_values("records", ascending=False)
)

print(
    "Distinct codes missing from core CPC reference:",
    cpc_code_review.loc[
        cpc_code_review["reference_match"] == "left_only",
        "cpc_code"
    ].nunique()
)

,cpc_code,commodities,records
34,01253.01,"[Onions and shallots, green]",141
73,01359.90,"[Other fruits, n.e., Other fruits, n.e.c.]",117
95,01520.01,"[Cassava, fresh]",115
138,21111.01,"[Meat of cattle with the bone, fresh or chille...",99
56,01342.01,[Pears],95
...,...,...,...
122,01919.92,"[Other legumes, for forage]",1
163,21439.04,[Juice of citrus fruit n.e.c.],1
162,21439.02,[Juice of lemon],1
143,21116i,"[Meat of goat, fresh or chilled (indigenous)]",1


Distinct codes missing from core CPC reference: 71


#### **Reviewing codes containing letters**



In [211]:
cpc_text = df_region["cpc_code"].astype("string").str.strip()

codes_containing_letters = df_region.loc[
    cpc_text.str.contains(r"[A-Za-z]", na=False),
    ["country", "region", "commodity", "cpc_code"]
].copy()

display(codes_containing_letters)

print("Records with alphabetic characters:", len(codes_containing_letters))

,country,region,commodity,cpc_code
9893,India,NaN,"Meat of goat, fresh or chilled (indigenous)",21116i
19441,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19463,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19483,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19503,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19523,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19543,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f


Records with alphabetic characters: 7


#### **Checking for potential loss of leading zeros**

In [212]:

cpc = df_region["cpc_code"].astype("string")
cpc_clean = cpc.str.strip()

# Allow digit groups with an optional decimal suffix.
# This checks syntax only, not official CPC validity.
valid_syntax = cpc_clean.str.fullmatch(
    r"\d+(?:\.\d+)?",
    na=False
)

invalid_cpc_format = df_region.loc[
    cpc.notna()
    & cpc_clean.ne("")
    & ~valid_syntax,
    ["country", "region", "commodity", "cpc_code"]
].copy()

display(invalid_cpc_format)

print("Invalid-looking code formats:", len(invalid_cpc_format))

,country,region,commodity,cpc_code
9893,India,NaN,"Meat of goat, fresh or chilled (indigenous)",21116i
19441,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19463,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19483,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19503,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19523,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f
19543,Panama,NaN,Raw cane or beet sugar (centrifugal only),2351f


Invalid-looking code formats: 7


#### **Checking for spaces and null/empty values**

In [213]:
cpc_text = df_region["cpc_code"].astype("string")

cpc_whitespace_issues = df_region.loc[
    cpc_text.notna()
    & (
        cpc_text.ne(cpc_text.str.strip())
        | cpc_text.str.contains(r"\s", regex=True, na=False)
    ),
    ["country", "region", "commodity", "cpc_code"]
].copy()

cpc_blank = df_region.loc[
    cpc_text.notna() & cpc_text.str.strip().eq(""),
    ["country", "region", "commodity", "cpc_code"]
].copy()

cpc_missing = df_region.loc[
    cpc_text.isna(),
    ["country", "region", "commodity", "cpc_code"]
].copy()

display(cpc_whitespace_issues)
display(cpc_blank)
display(cpc_missing)

print("Whitespace issues:", len(cpc_whitespace_issues))
print("Blank codes:", len(cpc_blank))
print("Missing codes:", len(cpc_missing))

,country,region,commodity,cpc_code


,country,region,commodity,cpc_code


,country,region,commodity,cpc_code


Whitespace issues: 0
Blank codes: 0
Missing codes: 0


#### **Checking spelling variations for the same code**

In [214]:
cpc_variants = (
    df_region.dropna(subset=["cpc_code"])
    .assign(
        _cpc_clean=lambda d: (
            d["cpc_code"].astype("string").str.strip()
        )
    )
    .groupby("_cpc_clean")
    .agg(
        original_codes=(
            "cpc_code",
            lambda s: sorted(s.astype(str).unique().tolist())
        ),
        number_of_variants=("cpc_code", "nunique"),
        total_records=("cpc_code", "size"),
        commodities=(
            "commodity",
            lambda s: sorted(s.dropna().astype(str).unique().tolist())
        )
    )
    .reset_index()
)

display(
    cpc_variants[cpc_variants["number_of_variants"] > 1]
    .sort_values("number_of_variants", ascending=False)
)

,_cpc_clean,original_codes,number_of_variants,total_records,commodities


#### **"Checking for many-to-many relationships" / "Detecting codes mapped to multiple commodities"**

In [215]:
cpc_commodity_mapping = (
    df_region.dropna(subset=["cpc_code", "commodity"])
    .groupby("cpc_code")
    .agg(
        commodities=(
            "commodity",
            lambda s: sorted(s.astype(str).unique().tolist())
        ),
        commodity_count=("commodity", "nunique"),
        records=("commodity", "size")
    )
    .reset_index()
)

display(
    cpc_commodity_mapping[
        cpc_commodity_mapping["commodity_count"] > 1
    ].sort_values("commodity_count", ascending=False)
)

,cpc_code,commodities,commodity_count,records
14,01199.90,"[Cereals n.e.c., Other cereals n.e.]",2,30
23,01231,"[Chillies and peppers, green (<i>Capsicum</i> ...",2,57
39,01290.90,"[Other vegetables, fresh n.e., Other vegetable...",2,52
42,01313,"[Plantains and cooking bananas, Plantains and ...",2,43
45,01316,"[Mangoes, guavas and mangosteens, Mangoes, gua...",2,302
48,01319,"[Other tropical and subtropical fruits, n.e., ...",2,60
53,01329,"[Other citrus fruit, n.e., Other citrus fruit,...",2,37
67,01353.01,"[Raspberries, Raspeberries]",2,36
70,01355.02,"[Cranberries, Cranberries and other fruits of ...",2,35
73,01359.90,"[Other fruits, n.e., Other fruits, n.e.c.]",2,117


#### **Checking for leading zeros**

In [216]:
cpc_codes = df_region["cpc_code"].dropna().astype("string").str.strip()

leading_zero_codes = cpc_codes[
    cpc_codes.str.match(r"^0+\d+(?:\.\d+)?$", na=False)
]

print("Codes beginning with zero:", len(leading_zero_codes))
display(leading_zero_codes.drop_duplicates().to_frame("cpc_code"))

Codes beginning with zero: 30719


,cpc_code
0,0111
1,0112
6,0113
7,0114
8,0115
...,...
27101,01919.08
27512,01708
27871,01216
28846,01990.01


### **Check on commodity**

### **Show missing values**

In [217]:
# Check missing and empty commodity values
commodity_clean = df_region["commodity"].astype("string").str.strip()

missing_commodity = df_region.loc[
    commodity_clean.isna() | commodity_clean.eq(""),
    ["country", "region", "commodity", "cpc_code"]
].copy()

print("Missing commodity records:", len(missing_commodity))
display(missing_commodity.head(20))

Missing commodity records: 0


,country,region,commodity,cpc_code


#### **Duplicate names with different formatting**

In [218]:
# Normalize names for comparison only
commodity_normalized = (
    df_region["commodity"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)

commodity_variants = (
    df_region.assign(_commodity_normalized=commodity_normalized)
    .dropna(subset=["_commodity_normalized"])
    .groupby("_commodity_normalized")
    .agg(
        original_names=("commodity", lambda s: sorted(s.astype(str).unique())),
        records=("commodity", "size"),
        cpc_codes=("cpc_code", lambda s: sorted(s.dropna().astype(str).unique()))
    )
    .reset_index()
)

commodity_variants = commodity_variants.loc[
    commodity_variants["original_names"].apply(len) > 1
].copy()

print("Names with formatting/case variants:", len(commodity_variants))
display(commodity_variants)

Names with formatting/case variants: 2


,_commodity_normalized,original_names,records,cpc_codes
35,cheese from whole cow milk,"[Cheese from Whole Cow Milk, Cheese from whole...",14,[22251.01]
171,"rice, milled","[Rice, Milled, Rice, milled]",37,[23161.02]


#### **Diff in Singular vs. plural**

In [219]:
# Install once if needed:
# %pip install inflect

import inflect

engine = inflect.engine()

commodity_names = (
    df_region["commodity"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

singular_plural_candidates = []

for name in commodity_names:
    singular = engine.singular_noun(name)

    if singular and singular.casefold() != name.casefold():
        if singular.casefold() in {
            value.casefold() for value in commodity_names
        }:
            singular_plural_candidates.append({
                "original_name": name,
                "possible_singular": singular
            })

singular_plural_review = pd.DataFrame(singular_plural_candidates)

display(singular_plural_review)

""


#### **Spelling errors**

In [220]:
commodity_frequency = (
    df_region["commodity"]
    .dropna()
    .astype("string")
    .str.strip()
    .value_counts()
    .rename_axis("commodity")
    .reset_index(name="records")
)

rare_commodity_names = commodity_frequency.loc[
    commodity_frequency["records"] <= 2
].copy()

print("Rare commodity names:", len(rare_commodity_names))
display(rare_commodity_names)

Rare commodity names: 28


,commodity,records
185,Pot barley,2
186,"Malt, whether or not roasted",2
187,"Other citrus fruit, n.e.",2
188,"Vegetables, dehydrated",2
189,"Coffee, decaffeinated or roasted",2
190,Cheese from whole cow milk,2
191,Natural honey,2
192,"Broad beans and horse beans, green",2
193,"Plums, dried",2
194,Food wastes,2


#### **General categories mixed with specific products**

In [221]:
# Potential broad or residual commodity categories
general_category_pattern = (
    r"\bother\b|"
    r"\bothers\b|"
    r"\bvarious\b|"
    r"\bmixed\b|"
    r"\bn\.e\.c\.?\b|"
    r"\bnot elsewhere classified\b"
)

commodity_text = df_region["commodity"].astype("string").str.strip()

general_commodity_candidates = df_region.loc[
    commodity_text.str.contains(
        general_category_pattern,
        case=False,
        regex=True,
        na=False
    ),
    ["country", "region", "commodity", "cpc_code"]
].copy()

print("Potential general-category records:", len(general_commodity_candidates))
display(general_commodity_candidates)

Potential general-category records: 669


,country,region,commodity,cpc_code
16,World,NaN,Cereals n.e.c.,01199.90
17,World,NaN,Cereals n.e.c.,01199.90
18,World,NaN,Cereals n.e.c.,01199.90
25,World,NaN,Mixed grain,01199.02
26,World,NaN,Mixed grain,01199.02
...,...,...,...,...
30515,Venezuela (Bolivarian Republic of),NaN,Plantains and others,01313
30553,Venezuela (Bolivarian Republic of),NaN,Plantains and others,01313
30591,Venezuela (Bolivarian Republic of),NaN,Plantains and others,01313
30630,Venezuela (Bolivarian Republic of),NaN,Plantains and others,01313


### **Check on year**

#### **Show missing values**

In [222]:
# Check missing or empty year values
year_clean = df_region["year"].astype("string").str.strip()

missing_year = df_region.loc[
    year_clean.isna() | year_clean.eq(""),
    ["country", "region", "commodity", "cpc_code", "year"]
].copy()

print("Missing year records:", len(missing_year))
display(missing_year)

Missing year records: 0


,country,region,commodity,cpc_code,year


#### **Non-numeric years**

In [223]:
# Identify years that cannot be interpreted as numeric values
year_numeric = pd.to_numeric(
    df_region["year"].astype("string").str.strip(),
    errors="coerce"
)

invalid_year_format = df_region.loc[
    df_region["year"].notna()
    & df_region["year"].astype("string").str.strip().ne("")
    & year_numeric.isna(),
    ["country", "region", "commodity", "cpc_code", "year"]
].copy()

print("Non-numeric year records:", len(invalid_year_format))
display(invalid_year_format)

Non-numeric year records: 0


,country,region,commodity,cpc_code,year


#### **Years outside the expected range**

In [224]:
from datetime import datetime

# FAO FLW database user guide documents records from 1965 onward.
# This is a dataset-level review boundary, not proof that older source studies are invalid.
MIN_EXPECTED_YEAR = 1965
MAX_EXPECTED_YEAR = datetime.now().year

year_numeric = pd.to_numeric(
    df_region["year"].astype("string").str.strip(),
    errors="coerce"
)

valid_numeric_year = year_numeric.notna() & np.isclose(
    year_numeric.fillna(-1),
    np.floor(year_numeric.fillna(-1))
)

outside_documented_range = df_region.loc[
    valid_numeric_year
    & (
        year_numeric.lt(MIN_EXPECTED_YEAR)
        | year_numeric.gt(MAX_EXPECTED_YEAR)
    ),
    ["country", "region", "commodity", "year", "reference", "url"]
].copy()

print(
    f"Review years outside {MIN_EXPECTED_YEAR}–{MAX_EXPECTED_YEAR}: "
    f"{len(outside_documented_range)}"
)
display(outside_documented_range.head(100))

Review years outside 1965–2026: 0


,country,region,commodity,year,reference,url


#### **Future or unusually old years**

In [225]:
# Future years
future_years = df_region.loc[
    valid_numeric_year & year_numeric.gt(MAX_EXPECTED_YEAR),
    ["country", "region", "commodity", "cpc_code", "year"]
].copy()

# Unusually old years: example threshold
VERY_OLD_YEAR_THRESHOLD = 1900

very_old_years = df_region.loc[
    valid_numeric_year & year_numeric.lt(VERY_OLD_YEAR_THRESHOLD),
    ["country", "region", "commodity", "cpc_code", "year"]
].copy()

print("Future year records:", len(future_years))
display(future_years)

print("Very old year records:", len(very_old_years))
display(very_old_years)

Future year records: 0


,country,region,commodity,cpc_code,year


Very old year records: 0


,country,region,commodity,cpc_code,year


### **Check on loss_percentage**

#### **Show missing values**

In [226]:
# Check missing and empty loss_percentage values
loss_text = df_region["loss_percentage"].astype("string").str.strip()

missing_loss_percentage = df_region.loc[
    loss_text.isna() | loss_text.eq(""),
    ["country", "region", "commodity", "year",
     "loss_percentage", "loss_percentage_original"]
].copy()

print("Missing loss_percentage records:", len(missing_loss_percentage))
display(missing_loss_percentage.head(20))

Missing loss_percentage records: 0


,country,region,commodity,year,loss_percentage,loss_percentage_original


#### **Non-numeric values**

In [227]:
# Convert values for validation only
loss_numeric = pd.to_numeric(loss_text, errors="coerce")

invalid_loss_format = df_region.loc[
    loss_text.notna()
    & loss_text.ne("")
    & loss_numeric.isna(),
    ["country", "region", "commodity", "year",
     "loss_percentage", "loss_percentage_original"]
].copy()

print("Non-numeric loss_percentage records:", len(invalid_loss_format))
display(invalid_loss_format.head(20))

Non-numeric loss_percentage records: 0


,country,region,commodity,year,loss_percentage,loss_percentage_original


#### **Values outside 0–100**

In [228]:
# Identify values outside the expected percentage range
out_of_range_loss = df_region.loc[
    loss_numeric.notna()
    & (
        loss_numeric.lt(0)
        | loss_numeric.gt(100)
    ),
    ["country", "region", "commodity", "year",
     "loss_percentage", "loss_percentage_original"]
].copy()

print("Out-of-range loss_percentage records:", len(out_of_range_loss))
display(out_of_range_loss.head(20))

Out-of-range loss_percentage records: 0


,country,region,commodity,year,loss_percentage,loss_percentage_original


#### **Outliers**

In [229]:
# Detect statistical outliers using the IQR method
loss_valid = loss_numeric.dropna()

if not loss_valid.empty:
    q1 = loss_valid.quantile(0.25)
    q3 = loss_valid.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    loss_outlier_mask = (
        loss_numeric.notna()
        & (
            loss_numeric.lt(lower_bound)
            | loss_numeric.gt(upper_bound)
        )
    )

    loss_percentage_outliers = df_region.loc[
        loss_outlier_mask,
        ["country", "region", "commodity", "year",
         "loss_percentage", "loss_percentage_original"]
    ].copy()

    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Potential outliers:", len(loss_percentage_outliers))

    display(loss_percentage_outliers.head(20))
else:
    print("No numeric values available for outlier detection.")

Q1: 1.49833
Q3: 4.55
IQR: 3.0516699999999997
Lower bound: -3.0791749999999993
Upper bound: 9.127505
Potential outliers: 3555


,country,region,commodity,year,loss_percentage,loss_percentage_original
8,World,NaN,Barley,2022,12.00,12%
26,World,NaN,Mixed grain,2018,20.03,20.03%
27,World,NaN,Mixed grain,2018,9.99,9.99%
28,World,NaN,Mixed grain,2018,13.61,13.61%
30,Myanmar,NaN,"Rice, milled",2015,11.77,11.77%
31,Myanmar,NaN,"Rice, milled",2015,17.65,17.65%
111,Burundi,NaN,Wheat,2019,14.50,14.5%
112,Burundi,NaN,Maize (corn),2019,18.00,18%
113,Burundi,NaN,Rice,2019,12.00,12%
564,Cambodia,NaN,Maize (corn),2020,30.00,30%


#### **Unit inconsistency**

In [230]:
# Flag small positive values for unit review
small_positive_loss = df_region.loc[
    loss_numeric.gt(0)
    & loss_numeric.lt(1),
    ["country", "region", "commodity", "year",
     "loss_percentage", "loss_percentage_original"]
].copy()

print("Positive values between 0 and 1:", len(small_positive_loss))
display(small_positive_loss.head(30))

Positive values between 0 and 1: 3191


,country,region,commodity,year,loss_percentage,loss_percentage_original
1,World,NaN,Maize (corn),2022,0.500000,0.5%
4,World,NaN,Maize (corn),2022,0.100000,0.1%
7,World,NaN,Sorghum,2022,0.500000,0.5%
555,Belarus,NaN,Raw milk of cattle,2007,0.010000,0.01%
556,Belarus,NaN,Raw milk of cattle,2006,0.020000,0.02%
557,Belarus,NaN,Raw milk of cattle,2005,0.020000,0.02%
558,Belarus,NaN,Raw milk of cattle,2004,0.020000,0.02%
559,Belarus,NaN,Raw milk of cattle,2003,0.020000,0.02%
560,Belarus,NaN,Raw milk of cattle,2002,0.020000,0.02%
561,Belarus,NaN,Raw milk of cattle,2001,0.040000,0.04%


#### **Compare with loss_percentage_original**

In [231]:
# Convert original values for comparison
original_text = (
    df_region["loss_percentage_original"]
    .astype("string")
    .str.strip()
)

original_numeric = pd.to_numeric(
    original_text,
    errors="coerce"
)

# Compare only rows where both values are numeric
comparable_mask = loss_numeric.notna() & original_numeric.notna()

# Exact numeric comparison
mismatch_mask = (
    comparable_mask
    & ~loss_numeric.eq(original_numeric)
)

loss_percentage_mismatch = df_region.loc[
    mismatch_mask,
    ["country", "region", "commodity", "year",
     "loss_percentage", "loss_percentage_original"]
].copy()

print("Comparable records:", int(comparable_mask.sum()))
print("Mismatched records:", len(loss_percentage_mismatch))
display(loss_percentage_mismatch.head(30))

Comparable records: 22025
Mismatched records: 0


,country,region,commodity,year,loss_percentage,loss_percentage_original


In [232]:
# Check whether values may differ by a factor of 100
factor_100_mask = (
    comparable_mask
    & (
        np.isclose(
            loss_numeric,
            original_numeric * 100,
            rtol=1e-6,
            atol=1e-8
        )
        | np.isclose(
            loss_numeric * 100,
            original_numeric,
            rtol=1e-6,
            atol=1e-8
        )
    )
    & ~np.isclose(
        loss_numeric,
        original_numeric,
        rtol=1e-6,
        atol=1e-8
    )
)

possible_unit_mismatch = df_region.loc[
    factor_100_mask,
    ["country", "region", "commodity", "year",
     "loss_percentage", "loss_percentage_original"]
].copy()

print("Possible factor-of-100 differences:", len(possible_unit_mismatch))
display(possible_unit_mismatch.head(30))

Possible factor-of-100 differences: 0


,country,region,commodity,year,loss_percentage,loss_percentage_original


### **Check on loss_percentage**

#### **Show missing values**

In [233]:
# Check missing and empty original loss percentage values
original_text = (
    df_region["loss_percentage_original"]
    .astype("string")
    .str.strip()
)

missing_original_loss = df_region.loc[
    original_text.isna() | original_text.eq(""),
    [
        "country", "region", "commodity", "year",
        "loss_percentage", "loss_percentage_original"
    ]
].copy()

print("Missing original values:", len(missing_original_loss))
display(missing_original_loss.head(20))####

Missing original values: 0


,country,region,commodity,year,loss_percentage,loss_percentage_original


#### **Non-numeric values**

In [234]:
# Attempt numeric conversion without changing the original column
original_numeric = pd.to_numeric(
    original_text,
    errors="coerce"
)

invalid_original_format = df_region.loc[
    original_text.notna()
    & original_text.ne("")
    & original_numeric.isna(),
    [
        "country", "region", "commodity", "year",
        "loss_percentage", "loss_percentage_original"
    ]
].copy()

print("Non-numeric original values:", len(invalid_original_format))
display(invalid_original_format.head(30))

Non-numeric original values: 9440


,country,region,commodity,year,loss_percentage,loss_percentage_original
0,World,NaN,Wheat,2022,1.00,1%
1,World,NaN,Maize (corn),2022,0.50,0.5%
2,World,NaN,Maize (corn),2022,2.00,2%
3,World,NaN,Maize (corn),2022,2.00,2%
4,World,NaN,Maize (corn),2022,0.10,0.1%
5,World,NaN,Maize (corn),2022,2.00,2%
6,World,NaN,Rice,2022,2.50,2.5%
7,World,NaN,Sorghum,2022,0.50,0.5%
8,World,NaN,Barley,2022,12.00,12%
9,World,NaN,Barley,2022,2.00,2%


#### **Percentage signs**

In [235]:
# Identify values containing a percentage sign
contains_percent_sign = original_text.str.contains(
    "%",
    regex=False,
    na=False
)

percentage_sign_records = df_region.loc[
    contains_percent_sign,
    [
        "country", "region", "commodity", "year",
        "loss_percentage", "loss_percentage_original"
    ]
].copy()

print("Values containing %:", len(percentage_sign_records))
display(percentage_sign_records.head(30))

Values containing %: 9025


,country,region,commodity,year,loss_percentage,loss_percentage_original
0,World,NaN,Wheat,2022,1.00,1%
1,World,NaN,Maize (corn),2022,0.50,0.5%
2,World,NaN,Maize (corn),2022,2.00,2%
3,World,NaN,Maize (corn),2022,2.00,2%
4,World,NaN,Maize (corn),2022,0.10,0.1%
5,World,NaN,Maize (corn),2022,2.00,2%
6,World,NaN,Rice,2022,2.50,2.5%
7,World,NaN,Sorghum,2022,0.50,0.5%
8,World,NaN,Barley,2022,12.00,12%
9,World,NaN,Barley,2022,2.00,2%


#### **Decimal formatting**

In [236]:
# Find original values containing commas
contains_comma = original_text.str.contains(
    ",",
    regex=False,
    na=False
)

comma_format_records = df_region.loc[
    contains_comma,
    [
        "country", "region", "commodity", "year",
        "loss_percentage", "loss_percentage_original"
    ]
].copy()

print("Values containing commas:", len(comma_format_records))
display(comma_format_records.head(30))

Values containing commas: 0


,country,region,commodity,year,loss_percentage,loss_percentage_original


#### **Compare original and cleaned values**

In [237]:
# Convert both columns for comparison only
loss_numeric = pd.to_numeric(
    df_region["loss_percentage"].astype("string").str.strip(),
    errors="coerce"
)

original_numeric = pd.to_numeric(
    original_text,
    errors="coerce"
)

comparable_mask = (
    loss_numeric.notna()
    & original_numeric.notna()
)

mismatch_mask = (
    comparable_mask
    & ~np.isclose(
        loss_numeric,
        original_numeric,
        rtol=1e-6,
        atol=1e-8
    )
)

original_vs_cleaned_mismatch = df_region.loc[
    mismatch_mask,
    [
        "country", "region", "commodity", "year",
        "loss_percentage", "loss_percentage_original"
    ]
].copy()

print("Comparable numeric records:", int(comparable_mask.sum()))
print("Numeric mismatches:", len(original_vs_cleaned_mismatch))
display(original_vs_cleaned_mismatch.head(30))

Comparable numeric records: 22025
Numeric mismatches: 0


,country,region,commodity,year,loss_percentage,loss_percentage_original


#### **Are the original values different by a factor of 100?**

In [238]:
# Detect possible factor-of-100 differences
factor_100_mask = (
    comparable_mask
    & (
        np.isclose(
            loss_numeric,
            original_numeric * 100,
            rtol=1e-6,
            atol=1e-8
        )
        | np.isclose(
            loss_numeric * 100,
            original_numeric,
            rtol=1e-6,
            atol=1e-8
        )
    )
    & ~np.isclose(
        loss_numeric,
        original_numeric,
        rtol=1e-6,
        atol=1e-8
    )
)

possible_factor_100_difference = df_region.loc[
    factor_100_mask,
    [
        "country", "region", "commodity", "year",
        "loss_percentage", "loss_percentage_original"
    ]
].copy()

print("Possible factor-of-100 differences:", len(possible_factor_100_difference))
display(possible_factor_100_difference.head(30))

Possible factor-of-100 differences: 0


,country,region,commodity,year,loss_percentage,loss_percentage_original


#### **Investigating the role of loss_percentage_original**

In [239]:
# Inspect both columns and their data types
print(df_region[
    ["loss_percentage", "loss_percentage_original"]
].dtypes)

display(
    df_region[
        ["loss_percentage", "loss_percentage_original"]
    ].sample(
        min(20, len(df_region)),
        random_state=42
    )
)

loss_percentage             float64
loss_percentage_original        str
dtype: object


,loss_percentage,loss_percentage_original
27126,1.50000,0–3%
17402,4.17000,4.17%
2320,2.50000,2.5
25378,3.60000,3.6
10883,4.00000,4
18329,2.00000,2
30801,2.61612,2.61612
6538,1.25000,1.25
21787,3.50000,3.5
16303,2.35000,2.35%


In [240]:
print("Original column dtype:", df_region["loss_percentage_original"].dtype)
print("Total records:", len(df_region))
print("Missing values:", df_region["loss_percentage_original"].isna().sum())
print("Unique non-null values:", df_region["loss_percentage_original"].nunique())

display(
    df_region["loss_percentage_original"]
    .value_counts(dropna=False)
    .head(20)
)

Original column dtype: str
Total records: 31465
Missing values: 0
Unique non-null values: 5995


loss_percentage_original
2.5        2309
3.5        1349
1          1280
2.65       1244
2           760
4           717
3.6         664
1.25        660
2.16667     660
1.32        619
4.40333     609
3.13714     609
6.42        562
4.55        541
1.3         330
4.43333     276
1.65        266
8           254
5           200
1%          178
Name: count, dtype: int64

### **Check on loss_quantity**

#### **Show missing values**

In [241]:
# Check missing and empty loss_quantity values
quantity_text = df_region["loss_quantity"].astype("string").str.strip()

missing_loss_quantity = df_region.loc[
    quantity_text.isna() | quantity_text.eq(""),
    ["country", "region", "commodity", "year", "loss_quantity"]
].copy()

print("Missing loss_quantity records:", len(missing_loss_quantity))
display(missing_loss_quantity.head(20))

Missing loss_quantity records: 26415


,country,region,commodity,year,loss_quantity
0,World,NaN,Wheat,2022,NaN
1,World,NaN,Maize (corn),2022,NaN
2,World,NaN,Maize (corn),2022,NaN
3,World,NaN,Maize (corn),2022,NaN
4,World,NaN,Maize (corn),2022,NaN
5,World,NaN,Maize (corn),2022,NaN
6,World,NaN,Rice,2022,NaN
7,World,NaN,Sorghum,2022,NaN
8,World,NaN,Barley,2022,NaN
9,World,NaN,Barley,2022,NaN


#### **Non-numeric values**

In [242]:
# Convert values for validation only
quantity_numeric = pd.to_numeric(
    quantity_text,
    errors="coerce"
)

invalid_quantity_format = df_region.loc[
    quantity_text.notna()
    & quantity_text.ne("")
    & quantity_numeric.isna(),
    ["country", "region", "commodity", "year", "loss_quantity"]
].copy()

print("Non-numeric loss_quantity records:", len(invalid_quantity_format))
display(invalid_quantity_format.head(30))

Non-numeric loss_quantity records: 691


,country,region,commodity,year,loss_quantity
25,World,NaN,Mixed grain,2018,60 million metric tons
29,Myanmar,NaN,"Rice, milled",2015,44.09kgs
30,Myanmar,NaN,"Rice, milled",2015,88.18kgs
31,Myanmar,NaN,"Rice, milled",2015,132.27kgs
32,Myanmar,NaN,"Rice, milled",2015,52.24kgs
33,Myanmar,NaN,"Rice, milled",2015,78.36kgs
34,Myanmar,NaN,"Rice, milled",2015,26.12kgs
576,Cambodia,NaN,Cauliflowers and broccoli,2006,52kg/MT
577,Cambodia,NaN,Cauliflowers and broccoli,2006,37kg/MT
578,Cambodia,NaN,Cauliflowers and broccoli,2006,31kg/MT


#### **Negative quantities**

In [243]:
# Identify negative quantities
negative_loss_quantity = df_region.loc[
    quantity_numeric.lt(0),
    ["country", "region", "commodity", "year", "loss_quantity"]
].copy()

print("Negative loss_quantity records:", len(negative_loss_quantity))
display(negative_loss_quantity.head(30))

Negative loss_quantity records: 0


,country,region,commodity,year,loss_quantity


#### **Zero values**

In [244]:
# Identify zero quantities without treating them as errors
zero_loss_quantity = df_region.loc[
    quantity_numeric.eq(0),
    ["country", "region", "commodity", "year", "loss_quantity"]
].copy()

print("Zero loss_quantity records:", len(zero_loss_quantity))
display(zero_loss_quantity.head(30))

Zero loss_quantity records: 56


,country,region,commodity,year,loss_quantity
998,Canada,NaN,Rye,2008,0
999,Canada,NaN,Oats,2008,0
1006,Canada,NaN,Rye,2007,0
1007,Canada,NaN,Oats,2007,0
1041,Canada,NaN,Rye,2006,0
1042,Canada,NaN,Oats,2006,0
1076,Canada,NaN,Rye,2005,0
1077,Canada,NaN,Oats,2005,0
1112,Canada,NaN,Rye,2004,0
2712,Cuba,NaN,Pomelos and grapefruits,2005,0


#### **Extreme quantities**

In [245]:
# Detect unusually large quantities using IQR
quantity_valid = quantity_numeric[
    quantity_numeric.notna() & quantity_numeric.ge(0)
]

if not quantity_valid.empty:
    q1 = quantity_valid.quantile(0.25)
    q3 = quantity_valid.quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    extreme_quantity_mask = (
        quantity_numeric.notna()
        & quantity_numeric.ge(0)
        & quantity_numeric.gt(upper_bound)
    )

    extreme_loss_quantity = df_region.loc[
        extreme_quantity_mask,
        ["country", "region", "commodity", "year", "loss_quantity"]
    ].copy()

    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Upper bound:", upper_bound)
    print("Potential extreme records:", len(extreme_loss_quantity))

    display(
        extreme_loss_quantity.sort_values(
            "loss_quantity",
            ascending=False
        ).head(30)
    )
else:
    print("No non-negative numeric quantities available.")

Q1: 1022.0
Q3: 36000.0
IQR: 34978.0
Upper bound: 88467.0
Potential extreme records: 601


,country,region,commodity,year,loss_quantity
19787,Peru,NaN,"Onions and shallots, dry (excluding dehydrated)",2004,99313
6233,France,NaN,Barley,2008,99000
7117,Germany,NaN,Maize (corn),2000,99000
18048,Nicaragua,NaN,Maize (corn),2004,98974
1592,Chad,NaN,Millet,2014,98697
3528,Denmark,NaN,Barley,2006,98300
2684,Cuba,NaN,Maize (corn),2007,98300
7095,Germany,NaN,Rye,2002,98000
7030,Germany,NaN,Rye,2008,98000
13703,Lithuania,NaN,Potatoes,2002,98000


#### **Unit inconsistency**

In [246]:
# Inspect numeric quantity distribution
print(
    quantity_numeric.describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

display(
    quantity_numeric.value_counts(dropna=False)
    .sort_index()
    .head(30)
)

count           4359.0
mean      64356.682888
std      218466.371092
min                0.0
1%                 0.0
5%                71.0
25%             1022.0
50%             7000.0
75%            36000.0
95%           250516.8
99%         1270395.62
max          3824000.0
Name: loss_quantity, dtype: Float64


loss_quantity
0.0          56
0.1           2
1.0           2
1.673         1
2.0           1
2.307245      1
3.0           5
4.0           2
6.0           2
8.0           1
10.0          3
11.0          4
12.0          1
12.303829     1
13.0          2
14.0          1
15.0          3
15.69393      1
17.0          2
18.0          7
19.0          2
20.0          4
21.0          1
22.0          3
23.0          3
24.0          7
25.0          1
28.0          1
29.0          4
30.0          2
Name: count, dtype: Int64

In [247]:
# Review the largest numeric quantities
largest_quantities = df_region.loc[
    quantity_numeric.notna(),
    ["country", "region", "commodity", "year", "loss_quantity"]
].copy()

largest_quantities["_quantity_numeric"] = quantity_numeric.loc[
    largest_quantities.index
]

largest_quantities = largest_quantities.sort_values(
    "_quantity_numeric",
    ascending=False
)

display(largest_quantities.head(30))

,country,region,commodity,year,loss_quantity,_quantity_numeric
26815,Ukraine,NaN,Potatoes,2014,3824000,3824000.0
26804,Ukraine,NaN,Potatoes,2015,3799000,3799000.0
20145,Poland,NaN,Potatoes,2000,3390000,3390000.0
26793,Ukraine,NaN,Potatoes,2016,3174000,3174000.0
20143,Poland,NaN,Potatoes,2001,2930000,2930000.0
20757,Russian Federation,NaN,Potatoes,2013,1983900,1983900.0
29511,United States of America,NaN,Potatoes,2000,1981700,1981700.0
20775,Russian Federation,NaN,Potatoes,2007,1825700,1825700.0
20141,Poland,NaN,Potatoes,2002,1817000,1817000.0
20760,Russian Federation,NaN,Potatoes,2012,1751700,1751700.0


### **Check on Activity**

#### **Check on missing values**

In [248]:
# Check missing or empty activity values
activity_text = df_region["activity"].astype("string").str.strip()

missing_activity = df_region.loc[
    activity_text.isna() | activity_text.eq(""),
    ["country", "region", "commodity", "cpc_code", "activity"]
].copy()

print("Missing activity records:", len(missing_activity))
display(missing_activity.head(20))

Missing activity records:

 7370


,country,region,commodity,cpc_code,activity
27,World,NaN,Mixed grain,01199.02,NaN
35,Myanmar,NaN,"Groundnuts, excluding shelled",0142,NaN
36,Myanmar,NaN,"Groundnuts, excluding shelled",0142,NaN
37,Myanmar,NaN,"Groundnuts, excluding shelled",0142,NaN
38,Myanmar,NaN,"Groundnuts, excluding shelled",0142,NaN
39,Myanmar,NaN,"Groundnuts, excluding shelled",0142,NaN
40,Myanmar,NaN,"Groundnuts, excluding shelled",0142,NaN
41,Myanmar,NaN,"Groundnuts, excluding shelled",0142,NaN
111,Burundi,NaN,Wheat,0111,NaN
112,Burundi,NaN,Maize (corn),0112,NaN


#### **Formatting inconsistencies**

In [249]:
# Normalize activity labels for comparison only
activity_normalized = (
    df_region["activity"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)

activity_format_review = (
    df_region.assign(_activity_normalized=activity_normalized)
    .dropna(subset=["_activity_normalized"])
    .groupby("_activity_normalized")
    .agg(
        original_labels=(
            "activity",
            lambda s: sorted(s.astype(str).unique().tolist())
        ),
        records=("activity", "size"),
        commodities=(
            "commodity",
            lambda s: sorted(s.dropna().astype(str).unique().tolist())
        )
    )
    .reset_index()
)

activity_format_review = activity_format_review.loc[
    activity_format_review["original_labels"].apply(len) > 1
].copy()

print("Potential formatting inconsistencies:", len(activity_format_review))
display(activity_format_review)

Potential formatting inconsistencies: 0


,_activity_normalized,original_labels,records,commodities


#### **Potential synonyms**

In [250]:
# List unique activity labels and their frequencies
activity_frequency = (
    df_region["activity"]
    .dropna()
    .astype("string")
    .str.strip()
    .value_counts()
    .rename_axis("activity")
    .reset_index(name="records")
)

print("Unique non-empty activity labels:", len(activity_frequency))
display(activity_frequency)

Unique non-empty activity labels: 180


,activity,records
0,Storage,6072
1,Transportation,5290
2,"Drying, Harvesting",3500
3,"Shelling, Threshing",3346
4,Winnowing,1356
...,...,...
175,"Drying, Harvesting, Milling, Sorting, Storage,...",1
176,"Sorting, Washing",1
177,"Milling, Storage",1
178,"Harvesting, Packaging, Sorting, Storage",1


In [251]:
# Install once if needed:
# %pip install rapidfuzz

from rapidfuzz import fuzz
from itertools import combinations

activity_names = activity_frequency["activity"].dropna().tolist()

similar_activity_candidates = []

for name1, name2 in combinations(activity_names, 2):
    similarity = fuzz.ratio(name1.casefold(), name2.casefold())

    if similarity >= 85:
        similar_activity_candidates.append({
            "activity_1": name1,
            "activity_2": name2,
            "similarity_score": similarity
        })

activity_similarity_review = pd.DataFrame(
    similar_activity_candidates,
    columns=["activity_1", "activity_2", "similarity_score"]
)

if not activity_similarity_review.empty:
    activity_similarity_review = activity_similarity_review.sort_values(
        "similarity_score",
        ascending=False
    )

print("Similar-label pairs:", len(activity_similarity_review))
display(activity_similarity_review.head(50))

Similar-label pairs: 60


,activity_1,activity_2,similarity_score
52,"Collection, Drying, Harvesting, Processing, Re...","Collection, Drying, Harvesting, Packaging, Pro...",95.278970
41,"Collection, Grading, Harvesting, Packaging, Re...","Collection, Harvesting, Packaging, Retailing, ...",94.674556
32,"Assembling, Grading, Harvesting, Loading, Sort...","Assembling, Grading, Harvesting, Sorting, Stor...",94.409938
59,"Drying, Harvesting, Milling, Storage, Threshin...","Drying, Harvesting, Milling, Sorting, Storage,...",94.267516
43,"Collection, Harvesting, Packaging, Sorting, Tr...","Collection, Drying, Harvesting, Packaging, Sor...",93.548387
42,"Collection, Grading, Harvesting, Packaging, Re...","Collection, Grading, Harvesting, Retailing, St...",93.413174
35,"Assembling, Grading, Harvesting, Loading, Sort...","Grading, Harvesting, Loading, Sorting, Storage...",92.405063
18,"Marketing, Storage, Transportation","Farm, Marketing, Storage, Transportation",91.891892
44,"Collection, Harvesting, Packaging, Sorting, Tr...","Collection, Harvesting, Packaging, Transportation",91.588785
27,"Collection, Grading, Harvesting, Packaging, Tr...","Collection, Harvesting, Packaging, Transportation",91.588785


#### **Spelling issues**

In [252]:
# Identify rarely used activity labels
rare_activity_labels = activity_frequency.loc[
    activity_frequency["records"] <= 2
].copy()

print("Rare activity labels:", len(rare_activity_labels))
display(rare_activity_labels)

Rare activity labels: 98


,activity,records
82,"Distribution, Retailing",2
83,"Storage, Trading",2
84,"Collection, Wholesale",2
85,"Distribution, Marketing, Storage",2
86,"Collection, Grading, Harvesting, Packaging, Tr...",2
...,...,...
175,"Drying, Harvesting, Milling, Sorting, Storage,...",1
176,"Sorting, Washing",1
177,"Milling, Storage",1
178,"Harvesting, Packaging, Sorting, Storage",1


#### **Broad vs. specific activities**

In [253]:
# Identify possible broad activity labels for manual review
broad_activity_pattern = (
    r"\bother\b|"
    r"\bothers\b|"
    r"\bgeneral\b|"
    r"\btotal\b|"
    r"\ball activities\b|"
    r"\bvarious\b|"
    r"\bunspecified\b|"
    r"\bmiscellaneous\b"
)

broad_activity_candidates = df_region.loc[
    activity_text.str.contains(
        broad_activity_pattern,
        case=False,
        regex=True,
        na=False
    ),
    ["country", "region", "commodity", "cpc_code", "activity"]
].copy()

print("Potential broad activity records:", len(broad_activity_candidates))
display(broad_activity_candidates.head(50))

Potential broad activity records: 0


,country,region,commodity,cpc_code,activity


#### **Checking compatibility between the activity and the commodity**

In [254]:
# Review observed activity-commodity combinations
activity_commodity_combinations = (
    df_region.groupby(
        ["activity", "commodity"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

print("Unique activity-commodity combinations:",
      len(activity_commodity_combinations))

display(activity_commodity_combinations.head(100))

Unique activity-commodity combinations: 1503


,activity,commodity,records
1098,Storage,Maize (corn),1596
1205,Transportation,Maize (corn),1299
1124,Storage,Rice,1071
1129,Storage,Sorghum,1042
1232,Transportation,Rice,997
...,...,...,...
998,Retailing,Potatoes,34
1448,NaN,Plantains and others,34
1484,NaN,Sugar cane,32
1444,NaN,"Pigeon peas, dry",32


In [255]:
# Replace "stage" with the actual column name in your dataset
activity_stage_combinations = (
    df_region.groupby(
        ["activity", "food_supply_stage"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

display(activity_stage_combinations.head(100))

,activity,food_supply_stage,records
283,NaN,Whole supply chain,5939
75,"Drying, Harvesting",Harvest,3500
255,Transportation,Farm,3367
227,"Shelling, Threshing",NaN,3342
237,Storage,Farm,3295
...,...,...,...
14,Cleaning,Farm,6
59,Curing,Farm,5
236,Stacking,Farm,5
73,Drying,Processing,5


### **Check on food_supply_stage**

#### **Show missing values**

In [256]:
# Check missing or empty food_supply_stage values
stage_text = df_region["food_supply_stage"].astype("string").str.strip()

missing_food_supply_stage = df_region.loc[
    stage_text.isna() | stage_text.eq(""),
    [
        "country", "region", "commodity",
        "activity", "food_supply_stage"
    ]
].copy()

print("Missing food_supply_stage records:", len(missing_food_supply_stage))
display(missing_food_supply_stage.head(20))

Missing food_supply_stage records: 3391


,country,region,commodity,activity,food_supply_stage
44,Burundi,NaN,Wheat,"Shelling, Threshing",NaN
47,Burundi,NaN,Maize (corn),"Shelling, Threshing",NaN
53,Burundi,NaN,Rice,"Shelling, Threshing",NaN
58,Burundi,NaN,Sorghum,"Shelling, Threshing",NaN
62,Burundi,NaN,Millet,"Shelling, Threshing",NaN
67,Burundi,NaN,Wheat,"Shelling, Threshing",NaN
70,Burundi,NaN,Maize (corn),"Shelling, Threshing",NaN
76,Burundi,NaN,Rice,"Shelling, Threshing",NaN
81,Burundi,NaN,Sorghum,"Shelling, Threshing",NaN
85,Burundi,NaN,Millet,"Shelling, Threshing",NaN


#### **Formatting inconsistencies**

In [257]:
# Normalize labels for comparison only
stage_normalized = (
    df_region["food_supply_stage"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)

stage_format_review = (
    df_region.assign(_stage_normalized=stage_normalized)
    .dropna(subset=["_stage_normalized"])
    .groupby("_stage_normalized")
    .agg(
        original_labels=(
            "food_supply_stage",
            lambda s: sorted(s.astype(str).unique().tolist())
        ),
        records=("food_supply_stage", "size"),
        activities=(
            "activity",
            lambda s: sorted(s.dropna().astype(str).unique().tolist())
        )
    )
    .reset_index()
)

stage_format_review = stage_format_review.loc[
    stage_format_review["original_labels"].apply(len) > 1
].copy()

print("Potential formatting inconsistencies:", len(stage_format_review))
display(stage_format_review)

Potential formatting inconsistencies: 0


,_stage_normalized,original_labels,records,activities


#### **Different phase names potentially referring to the same stage**

In [258]:
# List unique stage labels and frequencies
stage_frequency = (
    df_region["food_supply_stage"]
    .dropna()
    .astype("string")
    .str.strip()
    .value_counts()
    .rename_axis("food_supply_stage")
    .reset_index(name="records")
)

print("Unique stage labels:", len(stage_frequency))
display(stage_frequency)

Unique stage labels: 20


,food_supply_stage,records
0,Farm,10103
1,Whole supply chain,6147
2,Harvest,4160
3,Storage,3216
4,Transport,2091
5,Retail,754
6,Processing,441
7,Wholesale,325
8,Households,248
9,Post-harvest,168


In [259]:
# Install once if needed:
# %pip install rapidfuzz

from rapidfuzz import fuzz
from itertools import combinations

stage_names = stage_frequency["food_supply_stage"].tolist()

similar_stage_candidates = []

for stage1, stage2 in combinations(stage_names, 2):
    similarity = fuzz.ratio(stage1.casefold(), stage2.casefold())

    if similarity >= 85:
        similar_stage_candidates.append({
            "stage_1": stage1,
            "stage_2": stage2,
            "similarity_score": similarity
        })

stage_similarity_review = pd.DataFrame(
    similar_stage_candidates,
    columns=["stage_1", "stage_2", "similarity_score"]
)

if not stage_similarity_review.empty:
    stage_similarity_review = stage_similarity_review.sort_values(
        "similarity_score",
        ascending=False
    )

print("Similar stage-label pairs:", len(stage_similarity_review))
display(stage_similarity_review.head(50))

Similar stage-label pairs: 0


,stage_1,stage_2,similarity_score


#### **Broad vs. specific stages**

In [260]:
# Flag potentially broad or aggregate stage labels
broad_stage_pattern = (
    r"\bother\b|"
    r"\btotal\b|"
    r"\ball stages\b|"
    r"\bwhole supply chain\b|"
    r"\bvarious\b|"
    r"\bunspecified\b|"
    r"\bmultiple stages\b"
)

broad_stage_candidates = df_region.loc[
    stage_text.str.contains(
        broad_stage_pattern,
        case=False,
        regex=True,
        na=False
    ),
    [
        "country", "region", "commodity",
        "activity", "food_supply_stage"
    ]
].copy()

print("Potential broad-stage records:", len(broad_stage_candidates))
display(broad_stage_candidates.head(50))

Potential broad-stage records: 6147


,country,region,commodity,activity,food_supply_stage
35,Myanmar,NaN,"Groundnuts, excluding shelled",NaN,Whole supply chain
36,Myanmar,NaN,"Groundnuts, excluding shelled",NaN,Whole supply chain
37,Myanmar,NaN,"Groundnuts, excluding shelled",NaN,Whole supply chain
38,Myanmar,NaN,"Groundnuts, excluding shelled",NaN,Whole supply chain
39,Myanmar,NaN,"Groundnuts, excluding shelled",NaN,Whole supply chain
40,Myanmar,NaN,"Groundnuts, excluding shelled",NaN,Whole supply chain
41,Myanmar,NaN,"Groundnuts, excluding shelled",NaN,Whole supply chain
111,Burundi,NaN,Wheat,NaN,Whole supply chain
112,Burundi,NaN,Maize (corn),NaN,Whole supply chain
113,Burundi,NaN,Rice,NaN,Whole supply chain


#### **Checking compatibility between the food supply stage and the business activity**

In [261]:
# Review observed stage-activity combinations
stage_activity_combinations = (
    df_region.groupby(
        ["food_supply_stage", "activity"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

print("Unique stage-activity combinations:",
      len(stage_activity_combinations))

display(stage_activity_combinations.head(100))

Unique stage-activity combinations: 286


,food_supply_stage,activity,records
265,Whole supply chain,NaN,5939
101,Harvest,"Drying, Harvesting",3500
89,Farm,Transportation,3367
284,NaN,"Shelling, Threshing",3342
87,Farm,Storage,3295
...,...,...,...
77,Farm,Milling,6
279,Wholesale,"Sorting, Wholesale",5
215,Transport,"Marketing, Storage, Transportation",5
38,Farm,Curing,5


In [262]:
# Count activities associated with each supply-chain stage
stage_activity_summary = (
    df_region.groupby("food_supply_stage", dropna=False)
    .agg(
        records=("activity", "size"),
        unique_activities=("activity", "nunique"),
        activities=(
            "activity",
            lambda s: sorted(s.dropna().astype(str).unique().tolist())
        )
    )
    .reset_index()
    .sort_values("unique_activities", ascending=False)
)

display(stage_activity_summary)

,food_supply_stage,records,unique_activities,activities
3,Farm,10103,73,"[Assembling, Assembling, Farm, Assembling, Fie..."
18,Whole supply chain,6147,42,"[Assembling, Bagging, Grading, Harvesting, Loa..."
12,Processing,441,29,"[Bagging, Packaging, Cleaning, Sorting, Dewate..."
15,Storage,3216,19,"[Cleaning, Collection, Distribution, Drying, F..."
19,Wholesale,325,16,"[Collection, Marketing, Collection, Wholesale,..."
6,Harvest,4160,14,"[Cleaning, Drying, Field, Harvesting, Ripening..."
17,Transport,2091,13,"[Handling, Handling, Trading, Transportation, ..."
16,Trader,107,10,"[Collection, Collection, Marketing, Distributi..."
10,Post-harvest,168,9,"[Drying, Drying, Milling, Processing, Handling..."
8,Market,101,9,"[Collection, Distribution, Marketing, Wholesal..."


### **Check on treatment**

#### **Show missing values**

In [263]:
treatment_text = df_region["treatment"].astype("string").str.strip()

missing_treatment = df_region.loc[
    treatment_text.isna() | treatment_text.eq(""),
    [
        "country",
        "region",
        "commodity",
        "activity",
        "treatment"
    ]
].copy()

print("Number of missing treatment values:", len(missing_treatment))
display(missing_treatment.head(20))

Number of missing treatment values: 29678


,country,region,commodity,activity,treatment
0,World,NaN,Wheat,Processing,NaN
1,World,NaN,Maize (corn),Processing,NaN
2,World,NaN,Maize (corn),Processing,NaN
3,World,NaN,Maize (corn),Processing,NaN
4,World,NaN,Maize (corn),Processing,NaN
5,World,NaN,Maize (corn),Processing,NaN
6,World,NaN,Rice,Processing,NaN
7,World,NaN,Sorghum,Processing,NaN
8,World,NaN,Barley,Processing,NaN
9,World,NaN,Barley,Processing,NaN


#### **Detecting spelling and formatting variations**

In [264]:
treatment_check = df_region[["treatment"]].copy()

treatment_check["treatment_normalized"] = (
    treatment_check["treatment"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)

treatment_variants = (
    treatment_check
    .groupby("treatment_normalized", dropna=False)
    .agg(
        original_labels=(
            "treatment",
            lambda s: sorted(s.dropna().unique().tolist())
        ),
        records=("treatment", "size")
    )
    .reset_index()
)

treatment_format_review = treatment_variants[
    treatment_variants["original_labels"].apply(len) > 1
].copy()

display(treatment_format_review)

,treatment_normalized,original_labels,records
188,aman rice,"[Aman Rice, Aman rice]",63
200,aus rice,"[Aus Rice, Aus rice]",54
229,boro rice,"[Boro Rice, Boro rice]",63
395,indica rice,"[Indica Rice, Indica rice]",18
454,major season,"[Major Season, Major season]",99
458,manual harvesting,"[Manual Harvesting, Manual harvesting]",8
486,minor season,"[Minor Season, Minor season]",66
517,on-season fresh longan for export,"[On-season fresh longan for Export, On-season ...",4
622,starch factory,"[Starch Factory, Starch factory]",3
733,wheat b (naturally infested grain) during 90 d...,[Wheat B (Naturally infested grain) during 90...,2


#### **Displaying unique values and the frequency of each value**

In [265]:
treatment_frequency = (
    df_region["treatment"]
    .value_counts(dropna=False)
    .rename_axis("treatment")
    .reset_index(name="records")
)

display(treatment_frequency)

,treatment,records
0,NaN,29678
1,Boro rice,57
2,Aman rice,57
3,Major Season,54
4,Aus rice,48
...,...,...
760,"Barley during 90 days at 28°C, 70% RH and 12h/...",1
761,"Barley during 90 days at 28°C, 70% RH and 12h/...",1
762,"Barley during 90 days at 28°C, 70% RH and 12h/...",1
763,"Barley during 90 days at 28°C, 70% RH and 12h/...",1


In [266]:
unique_treatments = (
    df_region["treatment"]
    .dropna()
    .astype("string")
    .str.strip()
    .unique()
)

print("Number of unique treatment values:", len(unique_treatments))
print(sorted(unique_treatments.tolist()))

Number of unique treatment values: 764
[', 2 days ‘abusive’ temperature and humidity (25.1 ± 1.3 ◦C and 46.6 ± 6.0%RH)', '1 month storage in a Basket made from woven bamboo stem plastered with cow dung from inside (2–3 cm) and supported with PVC bag lining and with smoke infusion port from the bottom', '1 months storage in Bamboo stem basket plastered with cow dung from inside and supported with PVC plastic bag but without smoke infusion port', '1 months storage in Red clay brick structure internally plastered with cement (2 cm) and supported with PVC plastic bag but with no smoke infusion', '1 months storage in Red clay brick structure internally plastered with cement (2 cm) but with no PVC plastic bag lining and no smoke infusion', '1 months storage in Red clay brick wall internally plastered (2 cm cement) and supported with PVC plastic bag and smoke infusion port', '1 months storage in Storage in Red clay brick structure internally plastered with cement (2 cm) and supported with PVC

#### **Detecting textually similar values**

In [267]:
from itertools import combinations

treatment_labels = sorted(
    df_region["treatment"]
    .dropna()
    .astype("string")
    .str.strip()
    .loc[lambda s: s.ne("")]
    .unique()
    .tolist()
)

treatment_similarity_candidates = []

for value1, value2 in combinations(treatment_labels, 2):
    similarity = fuzz.ratio(
        value1.casefold(),
        value2.casefold()
    )

    if similarity >= 85:
        treatment_similarity_candidates.append({
            "treatment_1": value1,
            "treatment_2": value2,
            "similarity": similarity
        })

treatment_similarity_review = (
    pd.DataFrame(treatment_similarity_candidates)
    .sort_values("similarity", ascending=False)
    .reset_index(drop=True)
)

display(treatment_similarity_review)

,treatment_1,treatment_2,similarity
0,On-season fresh longan for Export,On-season fresh longan for export,100.000000
1,Major Season,Major season,100.000000
2,Minor Season,Minor season,100.000000
3,Boro Rice,Boro rice,100.000000
4,Aus Rice,Aus rice,100.000000
...,...,...,...
1454,6 months storage in Storage in Red clay brick ...,6 months storage in Storage in Red clay brick ...,85.049834
1455,3 months storage in Storage in Red clay brick ...,3 months storage in Storage in Red clay brick ...,85.049834
1456,Wheat B (Naturally infested grain) 90 days at ...,Wheat B (Naturally infested grain) during 90 d...,85.046729
1457,Indica Rice Variety,Japonica Rice Variety,85.000000


#### **Reviewing generic or ambiguous values**

In [268]:
treatment_broad_candidates = df_region.loc[
    df_region["treatment"]
    .astype("string")
    .str.contains(
        r"\b(other|others|various|unspecified|unknown|miscellaneous)\b",
        case=False,
        na=False,
        regex=True
    ),
    [
        "country",
        "region",
        "commodity",
        "activity",
        "treatment"
    ]
].copy()

display(treatment_broad_candidates)

C:\Users\GM\AppData\Local\Temp\ipykernel_30492\3890336587.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


,country,region,commodity,activity,treatment
2064,China,NaN,Maize (corn),Storage,Other storage methods
2076,China,NaN,Rice,Storage,Other storage methods - Indica rice
9871,India,NaN,Rice,Storage,Unspecified storage
25279,Brazil,NaN,Rice,Storage,Unspecified storage


#### **Reviewing the compatibility of treatment with activity and food_supply_stage**

In [269]:
treatment_activity_combinations = (
    df_region
    .groupby(
        ["treatment", "activity"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

display(treatment_activity_combinations)

,treatment,activity,records
1145,NaN,NaN,7200
1136,NaN,Storage,5340
1141,NaN,Transportation,5224
1041,NaN,"Drying, Harvesting",3498
1130,NaN,"Shelling, Threshing",3346
...,...,...,...
30,180 days storage in SuperGrain IV-R™ bag + 12%...,Storage,1
31,180 days storage in SuperGrain IV-R™ bag + 14%...,Storage,1
32,180 days storage in SuperGrain IV-R™ bag + 16%...,Storage,1
33,180 days storage in SuperGrain IV-R™ bag + 18%...,Storage,1


In [270]:
treatment_context_summary = (
    df_region
    .groupby(
        ["treatment", "activity", "food_supply_stage"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

display(treatment_context_summary)

,treatment,activity,food_supply_stage,records
1257,NaN,NaN,Whole supply chain,5902
1081,NaN,"Drying, Harvesting",Harvest,3498
1230,NaN,Transportation,Farm,3366
1208,NaN,"Shelling, Threshing",NaN,3342
1217,NaN,Storage,Farm,3273
...,...,...,...,...
9,1 months storage in Storage in Woven basket no...,Storage,Storage,1
8,1 months storage in Storage in Storage in wove...,Storage,Storage,1
7,1 months storage in Storage in Red clay brick ...,Storage,Storage,1
6,1 months storage in Storage in Red clay brick ...,Storage,Storage,1


### **Check on cause_of_loss**

#### **Show missing values**

In [271]:
cause_text = df_region["cause_of_loss"].astype("string").str.strip()

missing_cause_of_loss = df_region.loc[
    cause_text.isna() | cause_text.eq(""),
    [
        "country",
        "region",
        "commodity",
        "activity",
        "cause_of_loss"
    ]
].copy()

print("Number of missing causes:", len(missing_cause_of_loss))
display(missing_cause_of_loss.head(20))

Number of missing causes: 30367


,country,region,commodity,activity,cause_of_loss
0,World,NaN,Wheat,Processing,NaN
1,World,NaN,Maize (corn),Processing,NaN
2,World,NaN,Maize (corn),Processing,NaN
3,World,NaN,Maize (corn),Processing,NaN
4,World,NaN,Maize (corn),Processing,NaN
5,World,NaN,Maize (corn),Processing,NaN
6,World,NaN,Rice,Processing,NaN
7,World,NaN,Sorghum,Processing,NaN
8,World,NaN,Barley,Processing,NaN
9,World,NaN,Barley,Processing,NaN


#### **Discovering Cause Name Variations**

In [272]:
cause_check = df_region[["cause_of_loss"]].copy()

cause_check["cause_normalized"] = (
    cause_check["cause_of_loss"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)

cause_variants = (
    cause_check
    .groupby("cause_normalized", dropna=False)
    .agg(
        original_labels=(
            "cause_of_loss",
            lambda s: sorted(s.dropna().unique().tolist())
        ),
        records=("cause_of_loss", "size")
    )
    .reset_index()
)

cause_format_review = cause_variants[
    cause_variants["original_labels"].apply(len) > 1
].copy()

display(cause_format_review)

,cause_normalized,original_labels,records
120,insect damage,"[Insect Damage, Insect damage]",19
130,insects and rodents,"[Insects And Rodents, Insects and rodents]",2
195,mechanical damage,"[Mechanical Damage, Mechanical damage]",37
248,physical losses,"[Physical Losses, Physical losses]",94
279,pre-cooling and packaging,"[Pre-Cooling And Packaging, Pre-cooling and p...",2
309,"rodents, insects and moulds","[Rodents, Insects and Moulds, Rodents, Insects...",9
310,"rodents, insects, moulds","[Rodents, Insects, Moulds, Rodents, insects, m...",3
312,rotting,"[Rotting, rotting]",8
317,"rough handling, the effects of high ambient te...","[rough handling, the effects of high ambient t...",4
334,spreading out in sunlight,"[Spreading Out In Sunlight, Spreading out in s...",2


In [273]:
cause_frequency = (
    df_region["cause_of_loss"]
    .value_counts(dropna=False)
    .rename_axis("cause_of_loss")
    .reset_index(name="records")
)

display(cause_frequency)

,cause_of_loss,records
0,NaN,30367
1,Physical Losses,87
2,Insect pest damage,54
3,main causes of this were a mismatch in supply ...,41
4,Mechanical Damage,36
...,...,...
402,Carotenoid Loss During Drying,1
403,Price Discount For Every One Bruchid Hole Per ...,1
404,Insects and mold,1
405,"Insects, moisture and mold",1


#### **Detecting Multiple Causes in a Single Cell**

In [274]:
cause_text = df_region["cause_of_loss"].astype("string").str.strip()

multiple_cause_pattern = (
    r",|;|\||\s/\s|\s&\s|\band\b|\bor\b"
)

multiple_cause_mask = cause_text.str.contains(
    multiple_cause_pattern,
    case=False,
    na=False,
    regex=True
)

multiple_cause_candidates = df_region.loc[
    multiple_cause_mask,
    [
        "country",
        "region",
        "commodity",
        "activity",
        "cause_of_loss"
    ]
].copy()

print(
    "Potential records with multiple causes:",
    len(multiple_cause_candidates)
)

display(multiple_cause_candidates.head(30))

Potential records with multiple causes: 453


,country,region,commodity,activity,cause_of_loss
612,Cambodia,NaN,Cauliflowers and broccoli,NaN,Hot Weather During Harvest And Post Packaging
613,Cambodia,NaN,Cucumbers and gherkins,NaN,Hot Weather During Harvest And Post Packaging
614,Cambodia,NaN,Tomatoes,NaN,Hot Weather During Harvest And Post Packaging
615,Cambodia,NaN,"Other beans, green",NaN,Hot Weather During Harvest And Post Packaging
647,Cameroon,Boyo,Bananas,"Farm, Marketing, Storage, Transportation","Over-ripening, poor handling, poor storage"
648,Cameroon,Boyo,Bananas,Farm,"Over-ripening, poor handling, poor storage"
649,Cameroon,Boyo,Bananas,Transportation,"Over-ripening, poor handling"
650,Cameroon,Boyo,Bananas,"Processing, Ripening","Over-ripening, poor handling, poor storage"
651,Cameroon,Boyo,Bananas,Storage,"Over-ripening, poor handling, poor storage"
769,Cameroon,NaN,Plantains and cooking bananas,NaN,"rough handling, unprotected storage conditions..."


#### **Discovering Differences in Level of Detail / Granularity**

In [275]:
cause_granularity = df_region[["cause_of_loss"]].copy()

cause_granularity["cause_text"] = (
    cause_granularity["cause_of_loss"]
    .astype("string")
    .str.strip()
)

cause_granularity["word_count"] = (
    cause_granularity["cause_text"]
    .str.split()
    .str.len()
)

cause_granularity["character_count"] = (
    cause_granularity["cause_text"]
    .str.len()
)

display(
    cause_granularity[
        [
            "cause_of_loss",
            "word_count",
            "character_count"
        ]
    ].sort_values(
        "word_count",
        ascending=False,
        na_position="last"
    ).head(30)
)

,cause_of_loss,word_count,character_count
12504,The vast majority of mango farmers interviewed...,424,2594
27512,The study revealed that 70% of the surveyed fa...,268,1692
17875,Krishi Bazar Store Oranges In Cemented Floor O...,203,1267
7477,The results of the milling analysis have been ...,203,1282
3458,In the field 1.7 59% 33% During threshing 1.0...,187,1214
27494,Poor Post Harvest Management . Quantitative...,146,1040
27493,"Weather, Field Damage d. Quantitative post-...",145,1034
27495,. Quantitative post-harvest losses of economic...,142,1008
25116,Main harvest side flow reasons: Unacceptable s...,123,798
25114,Main harvest side flow reasons: Unacceptable s...,123,798


In [276]:
display(
    cause_granularity[
        ["word_count", "character_count"]
    ].describe()
)

,character_count
count,1098.0
mean,77.123862
std,171.5698
min,4.0
25%,15.0
50%,24.0
75%,63.0
max,2594.0


#### **Distinguishing Between Cause and Effect**

In [277]:
cause_outcome_pattern = (
    r"\b(loss|lost|waste|wasted|spoilage|spoiled|"
    r"damage|damaged|deterioration|deteriorated)\b"
)

cause_outcome_candidates = df_region.loc[
    df_region["cause_of_loss"]
    .astype("string")
    .str.contains(
        cause_outcome_pattern,
        case=False,
        na=False,
        regex=True
    ),
    [
        "country",
        "region",
        "commodity",
        "activity",
        "food_supply_stage",
        "cause_of_loss"
    ]
].copy()

display(cause_outcome_candidates.head(50))

C:\Users\GM\AppData\Local\Temp\ipykernel_30492\1323180583.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


,country,region,commodity,activity,food_supply_stage,cause_of_loss
569,Cambodia,"Kandal Province, and wholesalers and retailers...",Tomatoes,NaN,Farm,Immaturity Insect Damage Rotting/Decay
570,Cambodia,"Kandal Province, and wholesalers and retailers...",Tomatoes,NaN,Trader,Physical Damage Rotting/Decay
571,Cambodia,"Kandal Province, and wholesalers and retailers...",Tomatoes,NaN,Wholesale,Physical Damage Rotting/Decay
1960,Chile,North Patagonia Region,Pears,NaN,Farm,Insect Damage
2121,China,NaN,Wheat,NaN,Storage,decentralized agricultural system in China ce...
2122,China,NaN,Maize (corn),NaN,Storage,decentralized agricultural system in China ce...
2123,China,NaN,Rice,NaN,Storage,decentralized agricultural system in China ce...
2124,China,NaN,"Meat of cattle with the bone, fresh or chilled",NaN,Processing,decentralized agricultural system in China ce...
2125,China,NaN,"Meat of cattle with the bone, fresh or chilled",NaN,Storage,decentralized agricultural system in China ce...
2126,China,NaN,"Meat of cattle with the bone, fresh or chilled","Handling, Harvesting",Farm,decentralized agricultural system in China ce...


#### **Reviewing Causes by Commodity, Activity, and Stage**

In [278]:
cause_context_summary = (
    df_region
    .groupby(
        [
            "cause_of_loss",
            "commodity",
            "activity",
            "food_supply_stage"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

display(cause_context_summary.head(100))

,cause_of_loss,commodity,activity,food_supply_stage,records
1571,NaN,Maize (corn),Drying,Farm,830
1616,NaN,Maize (corn),Storage,Farm,829
1575,NaN,Maize (corn),"Drying, Harvesting",Harvest,818
1621,NaN,Maize (corn),Transportation,Farm,817
1614,NaN,Maize (corn),"Shelling, Threshing",NaN,814
...,...,...,...,...,...
971,NaN,Bananas,NaN,Whole supply chain,38
1678,NaN,"Mangoes, guavas and mangosteens",NaN,Whole supply chain,37
1491,NaN,"Groundnuts, excluding shelled",NaN,Whole supply chain,36
1754,NaN,Millet,NaN,Whole supply chain,35


### **Check on sample_size**

#### **Show missing values**

In [279]:
sample_text = df_region["sample_size"].astype("string").str.strip()

missing_sample_size = df_region.loc[
    sample_text.isna() | sample_text.eq(""),
    [
        "country",
        "region",
        "commodity",
        "year",
        "sample_size"
    ]
].copy()

print("Missing sample_size values:", len(missing_sample_size))
display(missing_sample_size.head(20))

Missing sample_size values: 29603


,country,region,commodity,year,sample_size
0,World,NaN,Wheat,2022,NaN
1,World,NaN,Maize (corn),2022,NaN
2,World,NaN,Maize (corn),2022,NaN
3,World,NaN,Maize (corn),2022,NaN
4,World,NaN,Maize (corn),2022,NaN
5,World,NaN,Maize (corn),2022,NaN
6,World,NaN,Rice,2022,NaN
7,World,NaN,Sorghum,2022,NaN
8,World,NaN,Barley,2022,NaN
9,World,NaN,Barley,2022,NaN


#### **Checking for Non-Numeric Values**

In [280]:
sample_numeric = pd.to_numeric(
    sample_text,
    errors="coerce"
)

invalid_sample_size = df_region.loc[
    sample_text.notna()
    & sample_text.ne("")
    & sample_numeric.isna(),
    [
        "country",
        "region",
        "commodity",
        "year",
        "sample_size"
    ]
].copy()

print("Non-numeric sample_size values:", len(invalid_sample_size))
display(invalid_sample_size.head(20))

Non-numeric sample_size values: 1206


,country,region,commodity,year,sample_size
568,Cambodia,"Kandal Province, and wholesalers and retailers...",Tomatoes,2013,onsite interviews were conducted involving 30 ...
569,Cambodia,"Kandal Province, and wholesalers and retailers...",Tomatoes,2013,onsite interviews were conducted involving 30 ...
570,Cambodia,"Kandal Province, and wholesalers and retailers...",Tomatoes,2013,onsite interviews were conducted involving 30 ...
571,Cambodia,"Kandal Province, and wholesalers and retailers...",Tomatoes,2013,onsite interviews were conducted involving 30 ...
647,Cameroon,Boyo,Bananas,2020,380 households
648,Cameroon,Boyo,Bananas,2020,380 households
649,Cameroon,Boyo,Bananas,2020,380 households
650,Cameroon,Boyo,Bananas,2020,380 households
651,Cameroon,Boyo,Bananas,2020,380 households
1298,Sri Lanka,Conventional value chain,Cabbages,2023,"1829 stakeholders (farmers, collectors, wholes..."


#### **Detecting Zero and Negative Values**

In [281]:
negative_sample_size = df_region.loc[
    sample_numeric < 0,
    [
        "country",
        "region",
        "commodity",
        "year",
        "sample_size"
    ]
].copy()

zero_sample_size = df_region.loc[
    sample_numeric == 0,
    [
        "country",
        "region",
        "commodity",
        "year",
        "sample_size"
    ]
].copy()

print("Negative values:", len(negative_sample_size))
display(negative_sample_size.head(20))

print("Zero values:", len(zero_sample_size))
display(zero_sample_size.head(20))

Negative values: 0


,country,region,commodity,year,sample_size


Zero values: 0


,country,region,commodity,year,sample_size


#### **Checking for Non-Integer Values / Decimals**

In [282]:
non_integer_mask = (
    sample_numeric.notna()
    & ~np.isclose(
        sample_numeric,
        np.round(sample_numeric),
        rtol=0,
        atol=1e-9
    )
)

non_integer_sample_size = df_region.loc[
    non_integer_mask,
    [
        "country",
        "region",
        "commodity",
        "year",
        "sample_size"
    ]
].copy()

print("Non-integer values:", len(non_integer_sample_size))
display(non_integer_sample_size.head(20))

Non-integer values: 2


,country,region,commodity,year,sample_size
26116,Uganda,NaN,Maize (corn),2016,1.193
26117,Uganda,NaN,Flour of triticale,2016,1.193


#### **Detecting Outliers**

In [283]:
valid_sample_sizes = sample_numeric.dropna()

if not valid_sample_sizes.empty:
    q1 = valid_sample_sizes.quantile(0.25)
    q3 = valid_sample_sizes.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    sample_size_outlier_mask = (
        (sample_numeric < lower_bound)
        | (sample_numeric > upper_bound)
    )

    sample_size_outliers = df_region.loc[
        sample_size_outlier_mask,
        [
            "country",
            "region",
            "commodity",
            "year",
            "sample_size"
        ]
    ].copy()

    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Potential outliers:", len(sample_size_outliers))

    display(
        sample_size_outliers.sort_values(
            "sample_size",
            ascending=False
        ).head(30)
    )
else:
    print("No numeric sample_size values available for outlier detection.")

Lower bound: -953.625
Upper bound: 1709.375
Potential outliers: 0


,country,region,commodity,year,sample_size


#### **Reviewing Statistical Distribution**

In [284]:
display(sample_numeric.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

count         656.0
mean     308.352723
std      374.717715
min           1.193
25%            45.0
50%           100.0
75%          710.75
90%           933.0
95%           933.0
99%           933.0
max           988.0
Name: sample_size, dtype: Float64

In [285]:
largest_sample_sizes = df_region.loc[
    sample_numeric.notna()
].assign(
    sample_size_numeric=sample_numeric.loc[sample_numeric.notna()]
).sort_values(
    "sample_size_numeric",
    ascending=False
)

display(largest_sample_sizes.head(20))

,m49_code,country,region,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,...,treatment,cause_of_loss,sample_size,method_data_collection,reference,url,notes,_region_normalized,_country_normalized,sample_size_numeric
8772,340,Honduras,"Choluteca, Copan, El Paraiso, Francisco Moraza...",0112,Maize (corn),2016,13.68,9.95-17.41,78.61-284.26 kg,NaN,...,NaN,NaN,988,Survey,Luciana Delgado; Monica Schuster-; Maximo Tore...,https://www.researchgate.net/publication/31868...,NaN,"choluteca, copan, el paraiso, francisco moraza...",honduras,988.0
16539,50,Bangladesh,Mymensingh,0113,Rice,2009,4.30,4.30%,NaN,Storage,...,Aus rice,NaN,933,Survey,B K Bala; M A Haque; Md. Anower Hossain; Shank...,NaN,NaN,mymensingh,bangladesh,933.0
16538,50,Bangladesh,Mymensingh,0113,Rice,2009,2.96,2.96%,NaN,Threshing,...,Aus rice,NaN,933,Survey,B K Bala; M A Haque; Md. Anower Hossain; Shank...,NaN,NaN,mymensingh,bangladesh,933.0
16537,50,Bangladesh,NaN,0113,Rice,2009,1.91,1.91%,NaN,Harvesting,...,Aus rice,NaN,933,Survey,B K Bala; M A Haque; Md. Anower Hossain; Shank...,NaN,NaN,NaN,bangladesh,933.0
16536,50,Bangladesh,Comilla,0113,Rice,2009,0.02,0.02%,NaN,Parboiling,...,Aman rice,NaN,933,Survey,B K Bala; M A Haque; Md. Anower Hossain; Shank...,NaN,NaN,comilla,bangladesh,933.0
16535,50,Bangladesh,NaN,0113,Rice,2009,3.70,3.70%,NaN,Storage,...,Aman rice,NaN,933,Survey,B K Bala; M A Haque; Md. Anower Hossain; Shank...,NaN,NaN,NaN,bangladesh,933.0
16534,50,Bangladesh,Mymensingh,0113,Rice,2009,2.23,2.23%,NaN,Threshing,...,Boro rice,NaN,933,Survey,B K Bala; M A Haque; Md. Anower Hossain; Shank...,NaN,NaN,mymensingh,bangladesh,933.0
16533,50,Bangladesh,Comilla,0113,Rice,2009,2.58,2.58%,NaN,Storage,...,Aus rice,NaN,933,Survey,B K Bala; M A Haque; Md. Anower Hossain; Shank...,NaN,NaN,comilla,bangladesh,933.0
16532,50,Bangladesh,NaN,0113,Rice,2009,0.17,0.17%,NaN,Wholesale,...,Aman rice,NaN,933,Survey,B K Bala; M A Haque; Md. Anower Hossain; Shank...,NaN,NaN,NaN,bangladesh,933.0
16531,50,Bangladesh,Comilla,0113,Rice,2009,6.79,6.79%,NaN,"Cleaning, Collection, Distribution, Drying, Fi...",...,Boro rice,NaN,933,Survey,B K Bala; M A Haque; Md. Anower Hossain; Shank...,NaN,NaN,comilla,bangladesh,933.0


### **Check on method_data_collection**

#### **Show missing values**

In [286]:
method_text = (
    df_region["method_data_collection"]
    .astype("string")
    .str.strip()
)

missing_method = df_region.loc[
    method_text.isna() | method_text.eq(""),
    [
        "country",
        "region",
        "commodity",
        "year",
        "method_data_collection"
    ]
].copy()

print("Missing method values:", len(missing_method))
display(missing_method.head(20))

Missing method values: 36


,country,region,commodity,year,method_data_collection
3677,El Salvador,NaN,Rice,2013,NaN
3678,El Salvador,NaN,Sorghum,2013,NaN
14163,Malawi,Zulu,"Groundnuts, excluding shelled",2015,NaN
14169,Malawi,Zulu,"Groundnuts, excluding shelled",2015,NaN
15974,Mexico,NaN,Wheat,2011,NaN
15975,Mexico,NaN,Maize (corn),2011,NaN
15976,Mexico,NaN,Maize (corn),2011,NaN
15977,Mexico,NaN,Rice,2011,NaN
15978,Mexico,NaN,Sorghum,2011,NaN
15979,Mexico,NaN,Barley,2011,NaN


#### **Displaying Method Names and Their Frequencies**

In [287]:
method_frequency = (
    df_region["method_data_collection"]
    .value_counts(dropna=False)
    .rename_axis("method_data_collection")
    .reset_index(name="records")
)

display(method_frequency)

,method_data_collection,records
0,Modelled Estimates,19497
1,FAO's annual Agriculture Production Questionna...,4209
2,Survey,3220
3,Literature Review,1694
4,National Accounts,1275
5,Controlled Experiment,629
6,Case Study,528
7,No Data Collection Specified,207
8,Expert Opinion,128
9,FLW Protocol,41


In [288]:
unique_methods = (
    df_region["method_data_collection"]
    .dropna()
    .astype("string")
    .str.strip()
    .unique()
)

print("Number of unique methods:", len(unique_methods))
print(sorted(unique_methods.tolist()))

Number of unique methods: 11
['Case Study', 'Census', 'Controlled Experiment', 'Expert Opinion', "FAO's annual Agriculture Production Questionnaires", 'FLW Protocol', 'Literature Review', 'Modelled Estimates', 'National Accounts', 'No Data Collection Specified', 'Survey']


#### **Detecting Formatting Variations in Method Names**

In [289]:
method_check = df_region[["method_data_collection"]].copy()

method_check["method_normalized"] = (
    method_check["method_data_collection"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)

method_variants = (
    method_check
    .groupby("method_normalized", dropna=False)
    .agg(
        original_labels=(
            "method_data_collection",
            lambda s: sorted(s.dropna().unique().tolist())
        ),
        records=("method_data_collection", "size")
    )
    .reset_index()
)

method_format_review = method_variants[
    method_variants["original_labels"].apply(len) > 1
].copy()

display(method_format_review)

,method_normalized,original_labels,records


#### **Detecting Similar Method Names Requiring Review**

In [290]:

method_labels = sorted(
    df_region["method_data_collection"]
    .dropna()
    .astype("string")
    .str.strip()
    .loc[lambda s: s.ne("")]
    .unique()
    .tolist()
)

method_similarity_candidates = []

for method1, method2 in combinations(method_labels, 2):
    similarity = fuzz.ratio(
        method1.casefold(),
        method2.casefold()
    )

    if similarity >= 85:
        method_similarity_candidates.append({
            "method_1": method1,
            "method_2": method2,
            "similarity": similarity
        })

method_similarity_review = pd.DataFrame(
    method_similarity_candidates,
    columns=["method_1", "method_2", "similarity"]
)

if not method_similarity_review.empty:
    method_similarity_review = (
        method_similarity_review
        .sort_values("similarity", ascending=False)
        .reset_index(drop=True)
    )

display(method_similarity_review)

,method_1,method_2,similarity


#### **Distinguishing Between Data Collection Methods and Loss Estimation Methods**

In [291]:
method_category_pattern = (
    r"\b(survey|questionnaire|interview|"
    r"measurement|measured|estimation|estimated|"
    r"model|statistics|statistical|administrative data)\b"
)

method_category_candidates = df_region.loc[
    df_region["method_data_collection"]
    .astype("string")
    .str.contains(
        method_category_pattern,
        case=False,
        na=False,
        regex=True
    ),
    [
        "country",
        "region",
        "commodity",
        "year",
        "method_data_collection"
    ]
].copy()

display(method_category_candidates.head(50))

C:\Users\GM\AppData\Local\Temp\ipykernel_30492\1573561479.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


,country,region,commodity,year,method_data_collection
0,World,NaN,Wheat,2022,Survey
1,World,NaN,Maize (corn),2022,Survey
2,World,NaN,Maize (corn),2022,Survey
3,World,NaN,Maize (corn),2022,Survey
4,World,NaN,Maize (corn),2022,Survey
5,World,NaN,Maize (corn),2022,Survey
6,World,NaN,Rice,2022,Survey
7,World,NaN,Sorghum,2022,Survey
8,World,NaN,Barley,2022,Survey
9,World,NaN,Barley,2022,Survey


#### **Reviewing Methods Based on Study Context**

In [292]:
method_context_columns = [
    col for col in [
        "method_data_collection",
        "country",
        "commodity",
        "activity",
        "food_supply_stage",
        "year"
    ]
    if col in df_region.columns
]

method_context_summary = (
    df_region
    .groupby(
        method_context_columns,
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

display(method_context_summary.head(100))

,method_data_collection,country,commodity,activity,food_supply_stage,year,records
26984,Survey,Bangladesh,Rice,Storage,Storage,2009,72
514,Controlled Experiment,Zimbabwe,"Cow peas, dry",Storage,Storage,2018,70
28439,Survey,Kenya,Maize (corn),Storage,Storage,2016,49
27123,Survey,China,Rice,Storage,Storage,2015,46
422,Controlled Experiment,Ethiopia,Maize (corn),Storage,Storage,2019,42
...,...,...,...,...,...,...,...
5086,Literature Review,Ethiopia,Mixed grain,NaN,Post-harvest,2014,6
28397,Survey,Indonesia,Rice,Drying,Farm,2012,6
502,Controlled Experiment,South Africa,Pears,Storage,Households,2018,6
26968,Survey,Bangladesh,Potatoes,Wholesale,Wholesale,2009,6


### **Check on refrence , url and notes**

In [293]:
from urllib.parse import urlparse

# ==================================================
# 1. Prepare temporary cleaned text for inspection
# ==================================================

columns_to_check = ["reference", "url", "notes"]

for col in columns_to_check:
    if col not in df_region.columns:
        raise KeyError(f"Column '{col}' does not exist in df_region.")

text_data = {}

for col in columns_to_check:
    text_data[col] = df_region[col].astype("string").str.strip()


# ==================================================
# 2. Missing values and blank strings
# ==================================================

missing_results = {}

for col in columns_to_check:
    missing_mask = (
        text_data[col].isna()
        | text_data[col].eq("").fillna(False)
    )

    missing_results[col] = df_region.loc[missing_mask].copy()

    print(f"\n{col}: {missing_mask.sum()} missing or blank values")
    display(missing_results[col].head(10))


# ==================================================
# 3. Exact duplicates in each column
# ==================================================

duplicate_results = {}

for col in columns_to_check:
    normalized = text_data[col].str.casefold()

    duplicate_mask = (
        normalized.notna()
        & normalized.ne("")
        & normalized.duplicated(keep=False)
    )

    duplicate_results[col] = df_region.loc[
        duplicate_mask
    ].copy()

    print(f"\n{col}: duplicate records")
    display(
        duplicate_results[col]
        .sort_values(col, na_position="last")
        .head(30)
    )


# ==================================================
# 4. Reference formatting variants
# ==================================================

reference_check = pd.DataFrame({
    "original_reference": df_region["reference"],
    "normalized_reference": (
        text_data["reference"]
        .str.casefold()
        .str.replace(r"\s+", " ", regex=True)
    )
})

reference_variants = (
    reference_check
    .dropna(subset=["normalized_reference"])
    .query("normalized_reference != ''")
    .groupby("normalized_reference")
    .agg(
        original_forms=(
            "original_reference",
            lambda s: sorted(s.dropna().unique().tolist())
        ),
        records=("original_reference", "size")
    )
    .reset_index()
)

reference_format_review = reference_variants[
    reference_variants["original_forms"].apply(len) > 1
].copy()

print("\nReference formatting variants:")
display(reference_format_review)


# ==================================================
# 5. URL validation candidates
# ==================================================

def check_url(value):
    if pd.isna(value) or not str(value).strip():
        return "Missing"

    value = str(value)

    # Detect leading/trailing whitespace in original value
    if value != value.strip():
        return "Whitespace around URL"

    # URLs should not contain embedded whitespace
    if re.search(r"\s", value):
        return "Whitespace inside URL"

    try:
        parsed = urlparse(value)

        if parsed.scheme not in ("http", "https"):
            return "Invalid or unsupported scheme"

        if not parsed.hostname:
            return "Missing hostname"

        if "." not in parsed.hostname and parsed.hostname != "localhost":
            return "Suspicious hostname"

        return "Format looks valid"

    except ValueError:
        return "Parsing error"


url_original = df_region["url"].astype("string")

url_validation = pd.DataFrame({
    "original_url": df_region["url"],
    "url_check": url_original.map(check_url)
})

url_review_mask = url_validation["url_check"].ne("Format looks valid")

url_review = df_region.loc[url_review_mask].copy()
url_review["url_check"] = url_validation.loc[
    url_review_mask, "url_check"
]

print("\nURL validation review:")
display(url_review.head(50))


# URLs repeated exactly after trimming whitespace
url_normalized = text_data["url"]

duplicate_url_mask = (
    url_normalized.notna()
    & url_normalized.ne("")
    & url_normalized.duplicated(keep=False)
)

duplicate_urls = df_region.loc[duplicate_url_mask].copy()

print("\nDuplicate URLs:")
display(duplicate_urls.head(50))


# ==================================================
# 6. Possible encoding problems in notes
# ==================================================

notes_text = text_data["notes"]

encoding_pattern = (
    r"Ã.|Â.|â€|â€™|â€œ|â€\x9d|"
    r"ï¿½|�"
)

encoding_mask = notes_text.str.contains(
    encoding_pattern,
    case=False,
    na=False,
    regex=True
)

possible_encoding_issues = df_region.loc[
    encoding_mask,
    ["notes"]
].copy()

print("\nPossible encoding issues in notes:")
display(possible_encoding_issues.head(50))


# ==================================================
# 7. Notes that may contain structured information
# ==================================================

# Search for years, URLs, email addresses, and common labels.
# These are review candidates, not confirmed data-quality errors.

structured_info_pattern = (
    r"\b(?:19|20)\d{2}\b"             # Years
    r"|https?://"                     # URLs
    r"|\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b"  # Emails
    r"|\bcountry\s*:"
    r"|\bcommodity\s*:"
    r"|\bregion\s*:"
    r"|\bmethod\s*:"
)

structured_notes_mask = notes_text.str.contains(
    structured_info_pattern,
    case=False,
    na=False,
    regex=True
)

structured_notes_candidates = df_region.loc[
    structured_notes_mask,
    ["country", "commodity", "year", "notes"]
].copy()

# Keep only columns that actually exist
structured_notes_candidates = structured_notes_candidates[
    [c for c in structured_notes_candidates.columns
     if c in df_region.columns]
]

print("\nNotes containing possible structured information:")
display(structured_notes_candidates.head(50))


# ==================================================
# 8. Summary of checks
# ==================================================

quality_summary = pd.DataFrame({
    "column": columns_to_check,
    "missing_or_blank": [
        len(missing_results[col]) for col in columns_to_check
    ],
    "duplicate_records": [
        len(duplicate_results[col]) for col in columns_to_check
    ]
})

display(quality_summary)


reference: 73 missing or blank values


,m49_code,country,region,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,food_supply_stage,treatment,cause_of_loss,sample_size,method_data_collection,reference,url,notes,_region_normalized,_country_normalized
2220,170,Colombia,NaN,01317,Papayas,2016,3.00,3,NaN,NaN,Retail,NaN,NaN,NaN,No Data Collection Specified,NaN,NaN,NaN,NaN,colombia
2221,170,Colombia,NaN,01317,Papayas,2016,3.10,3.1,NaN,NaN,Storage,NaN,Conditioning,NaN,No Data Collection Specified,NaN,NaN,NaN,NaN,colombia
2222,170,Colombia,NaN,01317,Papayas,2016,17.67,17.67,NaN,NaN,Harvest,NaN,NaN,NaN,No Data Collection Specified,NaN,NaN,NaN,NaN,colombia
3577,218,Ecuador,NaN,0112,Maize (corn),2013,20.00,10.0-30,NaN,NaN,NaN,NaN,NaN,NaN,Expert Opinion,NaN,http://www.forumdofuturo.com.br/site/wp-conten...,url not found,NaN,ecuador
3578,218,Ecuador,NaN,01194,Quinoa,2013,5.00,5,NaN,NaN,Pre-harvest,NaN,Rain Pre-Harvest Causing Germination On The Stalk,NaN,Expert Opinion,NaN,http://www.forumdofuturo.com.br/site/wp-conten...,url not found,NaN,ecuador
3579,218,Ecuador,NaN,01701,"Beans, dry",2013,10.00,10,NaN,NaN,Farm,NaN,Threshing,NaN,Expert Opinion,NaN,http://www.forumdofuturo.com.br/site/wp-conten...,url not found,NaN,ecuador
3580,218,Ecuador,NaN,01701,"Beans, dry",2013,10.00,10,NaN,NaN,Storage,NaN,Rodents,NaN,Expert Opinion,NaN,http://www.forumdofuturo.com.br/site/wp-conten...,url not found,NaN,ecuador
3676,222,El Salvador,NaN,0112,Maize (corn),2013,8.00,8,NaN,NaN,Farm,NaN,Small-Scale Corn And Bean Farmers Account For ...,NaN,Expert Opinion,NaN,http://www.forumdofuturo.com.br/site/wp-conten...,url not found,NaN,el salvador
3677,222,El Salvador,NaN,0113,Rice,2013,6.00,6,NaN,NaN,Farm,NaN,Small-Scale Corn And Bean Farmers Account For ...,NaN,NaN,NaN,http://www.forumdofuturo.com.br/site/wp-conten...,NaN,NaN,el salvador
3678,222,El Salvador,NaN,0114,Sorghum,2013,6.00,6,NaN,NaN,Farm,NaN,Small-Scale Corn And Bean Farmers Account For ...,NaN,NaN,NaN,http://www.forumdofuturo.com.br/site/wp-conten...,NaN,NaN,el salvador



url: 9342 missing or blank values


,m49_code,country,region,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,food_supply_stage,treatment,cause_of_loss,sample_size,method_data_collection,reference,url,notes,_region_normalized,_country_normalized
0,1,World,NaN,0111,Wheat,2022,1.0,1%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is wheat flour,NaN,world
1,1,World,NaN,0112,Maize (corn),2022,0.5,0.5%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is maize flour.,NaN,world
2,1,World,NaN,0112,Maize (corn),2022,2.0,2%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is Maize (corn) grits.,NaN,world
3,1,World,NaN,0112,Maize (corn),2022,2.0,2%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is Maize (corn) grits flaked.,NaN,world
4,1,World,NaN,0112,Maize (corn),2022,0.1,0.1%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is corn starch.,NaN,world
5,1,World,NaN,0112,Maize (corn),2022,2.0,2%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is undenaturated ethyl alcohol.,NaN,world
6,1,World,NaN,0113,Rice,2022,2.5,2.5%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is milled rice.,NaN,world
7,1,World,NaN,0114,Sorghum,2022,0.5,0.5%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is flour of sorghum.,NaN,world
8,1,World,NaN,0115,Barley,2022,12.0,12%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is barley malt,NaN,world
9,1,World,NaN,0115,Barley,2022,2.0,2%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is pot barley,NaN,world



notes: 29931 missing or blank values


...


Possible encoding issues in notes:


,notes



Notes containing possible structured information:


,country,commodity,year,notes
551,Western Africa,Rice,2010,Sourced from FAO 2007
552,Western Africa,Rice,2010,Sourced from FAO 2007
553,Western Africa,Rice,2010,Sourced from FAO 2007
554,Western Africa,Rice,2010,Sourced from FAO 2007
566,Cambodia,Tomatoes,2015,Sourced from Weinberger et al 2008
567,Cambodia,"Beans, dry",2015,Sourced from Weinberger et al 2008
572,Cambodia,Cauliflowers and broccoli,2008,Sourced from Weinberger et al 2008
573,Cambodia,Cucumbers and gherkins,2008,Sourced from Weinberger et al 2008
616,Algeria,Potatoes,2005,Sourced from Yahia 2005
975,Canada,Mixed grain,2014,"Sourced from Gunders, (2012)"


,column,missing_or_blank,duplicate_records
0,reference,73,31348
1,url,9342,22097
2,notes,29931,1225


---
# Final validation summary and reproducible outputs

This section consolidates key checks into machine-readable outputs. It does not change `world_food_loss`; all normalized fields are for comparison only.

## Authoritative references

- FAO Food Loss and Waste Database: https://www.fao.org/platform-food-loss-waste/flw-data/en/
- FAO FLW Database User Guide: https://www.fao.org/platform-food-loss-waste/flw-data/user-guide/en
- UN M49 Overview: https://unstats.un.org/unsd/methodology/m49/overview/
- UN M49 Methodology: https://unstats.un.org/unsd/methodology/m49/
- UN CPC 2.1 classification: https://unstats.un.org/unsd/classifications/Econ/Download/In%20Text/CPC_Ver_2.1_Exp_Notes_Updated_3Apr2025.xlsx
- FAO Caliper classification downloads and mappings: https://www.fao.org/statistics/caliper/tools/download/en

**Reference policy:** UN M49 is authoritative for M49 codes and its geographic hierarchy; CPC is authoritative for CPC classification; FAO documentation defines the meaning and scope of the source dataset. GitHub datasets can be useful cross-checks, but should not override the official reference without documented justification.

## How to interpret flags

- `FAIL` / `MISMATCH`: a rule is violated and needs correction or a documented exception.
- `REVIEW`: the value is unusual or context-dependent; do not automatically modify it.
- `NOT_COMPARABLE`: insufficient information for that specific check.
- A valid code does not guarantee that the code is paired with the right country, region, commodity, or unit.


In [295]:
# Build final audit summary and export review tables.
# This cell is defensive: it only uses outputs that exist and never overwrites source data.

from pathlib import Path
from datetime import datetime, timezone
import json

REPORT_DIR = Path("../report") / "step_1_validation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def safe_write_csv(frame, filename):
    if isinstance(frame, pd.DataFrame):
        path = REPORT_DIR / filename
        frame.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"Saved {path.resolve()} ({len(frame):,} rows)")
        return path
    print(f"Skipped {filename}: expected DataFrame, got {type(frame).__name__}")
    return None

# Overall profile: one row per source column.
profile_rows = []
for column in world_food_loss.columns:
    series = world_food_loss[column]
    text_values = series.astype("string") if (
        pd.api.types.is_object_dtype(series.dtype)
        or pd.api.types.is_string_dtype(series.dtype)
    ) else None
    blank_count = int(text_values.str.strip().eq("").fillna(False).sum()) if text_values is not None else 0
    profile_rows.append({
        "column": column,
        "dtype": str(series.dtype),
        "row_count": int(len(series)),
        "missing_count": int(series.isna().sum()),
        "missing_pct": round(float(series.isna().mean() * 100), 4) if len(series) else 0,
        "blank_string_count": blank_count,
        "distinct_non_null": int(series.nunique(dropna=True)),
    })
final_column_profile = pd.DataFrame(profile_rows)

# Whole-row duplicates and likely duplicate source records are different concepts.
exact_duplicate_rows = world_food_loss.loc[
    world_food_loss.duplicated(keep=False)
].copy()

key_columns = [
    col for col in ["country", "region", "cpc_code", "commodity", "year",
                    "activity", "food_supply_stage", "reference"]
    if col in world_food_loss.columns
]
if key_columns:
    duplicate_candidate_rows = world_food_loss.loc[
        world_food_loss.duplicated(subset=key_columns, keep=False)
    ].copy()
else:
    duplicate_candidate_rows = pd.DataFrame()

# Compact validation status summary if the earlier validation cell ran.
status_summary = pd.DataFrame()
if "checked" in globals() and isinstance(checked, pd.DataFrame):
    for status_col in ["country_code_status", "region_review_status"]:
        if status_col in checked.columns:
            part = checked[status_col].value_counts(dropna=False).rename_axis("status").reset_index(name="records")
            part.insert(0, "validation", status_col)
            status_summary = pd.concat([status_summary, part], ignore_index=True)

# Export core audit outputs.
safe_write_csv(final_column_profile, "column_profile.csv")
safe_write_csv(exact_duplicate_rows, "exact_duplicate_rows.csv")
safe_write_csv(duplicate_candidate_rows, "duplicate_record_candidates.csv")
safe_write_csv(status_summary, "geographic_validation_status_summary.csv")

# Export any available review tables from the notebook.
review_tables = {
    "country_code_issues.csv": "country_code_issues",
    "region_level_review.csv": "region_level_review",
    "unexpected_region_report.csv": "unexpected_region_report",
    "cpc_code_review.csv": "cpc_code_review",
    "reference_format_review.csv": "reference_format_review",
    "out_of_range_loss_percentage.csv": "out_of_range_loss",
    "negative_loss_quantity.csv": "negative_loss_quantity",
    "invalid_sample_size.csv": "invalid_sample_size",
}
for filename, variable_name in review_tables.items():
    value = globals().get(variable_name)
    if isinstance(value, pd.DataFrame):
        safe_write_csv(value, filename)

run_manifest = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": str(globals().get("DATA_PATH", "path not recorded")),
    "source_rows": int(len(world_food_loss)),
    "source_columns": int(len(world_food_loss.columns)),
    "exact_duplicate_rows_including_all_copies": int(world_food_loss.duplicated(keep=False).sum()),
    "key_columns_used_for_duplicate_candidates": key_columns,
    "source_modified": False,
    "reference_links": {
        "FAO_FLW": "https://www.fao.org/platform-food-loss-waste/flw-data/en/",
        "FAO_FLW_User_Guide": "https://www.fao.org/platform-food-loss-waste/flw-data/user-guide/en",
        "UN_M49": "https://unstats.un.org/unsd/methodology/m49/overview/",
        "UN_CPC_2_1": "https://unstats.un.org/unsd/classifications/Econ/Download/In%20Text/CPC_Ver_2.1_Exp_Notes_Updated_3Apr2025.xlsx",
    },
}
manifest_path = REPORT_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {manifest_path.resolve()}")

print("\nFINAL AUDIT OVERVIEW")
display(pd.DataFrame([
    {"metric": "Rows", "value": len(world_food_loss)},
    {"metric": "Columns", "value": len(world_food_loss.columns)},
    {"metric": "Exact duplicate rows (including all copies)", "value": int(world_food_loss.duplicated(keep=False).sum())},
    {"metric": "Rows with at least one missing value", "value": int(world_food_loss.isna().any(axis=1).sum())},
    {"metric": "Rows with no missing values", "value": int((~world_food_loss.isna().any(axis=1)).sum())},
]))
print("\nReview files are saved under:", REPORT_DIR.resolve())


Saved D:\My_projects\data-engineer\waste-food-project\report\step_1_validation\column_profile.csv (18 rows)
Saved D:\My_projects\data-engineer\waste-food-project\report\step_1_validation\exact_duplicate_rows.csv (339 rows)
Saved D:\My_projects\data-engineer\waste-food-project\report\step_1_validation\duplicate_record_candidates.csv (2,862 rows)
Saved D:\My_projects\data-engineer\waste-food-project\report\step_1_validation\geographic_validation_status_summary.csv (4 rows)
Saved D:\My_projects\data-engineer\waste-food-project\report\step_1_validation\country_code_issues.csv (282 rows)
Saved D:\My_projects\data-engineer\waste-food-project\report\step_1_validation\region_level_review.csv (666 rows)
Saved D:\My_projects\data-engineer\waste-food-project\report\step_1_validation\unexpected_region_report.csv (378 rows)
Saved D:\My_projects\data-engineer\waste-food-project\report\step_1_validation\cpc_code_review.csv (196 rows)
Saved D:\My_projects\data-engineer\waste-food-project\report\step_1

,metric,value
0,Rows,31465
1,Columns,18
2,Exact duplicate rows (including all copies),339
3,Rows with at least one missing value,31465
4,Rows with no missing values,0



Review files are saved under: D:\My_projects\data-engineer\waste-food-project\report\step_1_validation
